In [12]:
# Cell 1: Create a 4-channel WAV file with correct sample rate
import soundfile as sf
import sounddevice as sd
import numpy as np
import os
import pathlib

# File paths
looming_path = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio\participant_00_design_looming.wav"
tactile_path = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio\participant_00_design_tactile.wav"
out_dir = r"C:\Users\cogpsy-vrlab\Desktop\audio sample"
out_file = "participant_00_design_4ch.wav"

# Create output directory if it doesn't exist
os.makedirs(out_dir, exist_ok=True)
out_path = pathlib.Path(out_dir) / out_file

# Check available devices and their sample rates
print("=== CHECKING AUDIO DEVICES ===")
output_12_id = 25  # Output 1/2 (2- Komplete Audio 6 MK2)
output_34_id = 22  # Output 3/4 (2- Komplete Audio 6 MK2)

try:
    device_12_info = sd.query_devices(output_12_id)
    device_34_info = sd.query_devices(output_34_id)
    
    print(f"Device {output_12_id}: {device_12_info['name']}")
    print(f"  Default sample rate: {device_12_info['default_samplerate']} Hz")
    
    print(f"Device {output_34_id}: {device_34_info['name']}")
    print(f"  Default sample rate: {device_34_info['default_samplerate']} Hz")
    
    # Determine target sample rate from devices
    target_sr = int(device_12_info['default_samplerate'])
    print(f"Target sample rate: {target_sr} Hz")
    
except Exception as e:
    print(f"Could not query devices: {e}")
    print("Defaulting to 44100 Hz (standard rate)")
    target_sr = 44100

# 1) Read both input files
print("\n=== READING INPUT FILES ===")
loom, sr1 = sf.read(looming_path, always_2d=True)
tact, sr2 = sf.read(tactile_path, always_2d=True)

if sr1 != sr2:
    print(f"Warning: Sample rate mismatch between input files ({sr1} vs {sr2})")
    print(f"Will use higher sample rate: {max(sr1, sr2)} Hz")
    sr = max(sr1, sr2)
else:
    sr = sr1

print(f"Input file sample rate: {sr} Hz")
print(f"Looming file shape: {loom.shape}")
print(f"Tactile file shape: {tact.shape}")

# 2) Resample if needed (simple implementation without scipy)
if sr != target_sr:
    print(f"\n=== RESAMPLING FROM {sr} Hz TO {target_sr} Hz ===")
    
    # Function to resample a single channel using NumPy
    def resample_channel(audio_channel, orig_sr, target_sr):
        # Calculate the ratio between target and original sample rates
        ratio = target_sr / orig_sr
        
        # Calculate new length
        new_length = int(len(audio_channel) * ratio)
        
        # Create time arrays for interpolation
        orig_time = np.linspace(0, 1, len(audio_channel))
        new_time = np.linspace(0, 1, new_length)
        
        # Interpolate
        return np.interp(new_time, orig_time, audio_channel)
    
    # Resample each channel separately
    loom_resampled = np.zeros((int(len(loom) * target_sr / sr), loom.shape[1]))
    tact_resampled = np.zeros((int(len(tact) * target_sr / sr), tact.shape[1]))
    
    for i in range(loom.shape[1]):
        loom_resampled[:, i] = resample_channel(loom[:, i], sr, target_sr)
    
    for i in range(tact.shape[1]):
        tact_resampled[:, i] = resample_channel(tact[:, i], sr, target_sr)
    
    loom = loom_resampled
    tact = tact_resampled
    sr = target_sr
    
    print(f"After resampling - Looming shape: {loom.shape}")
    print(f"After resampling - Tactile shape: {tact.shape}")

# 3) Pad shorter file with zeros so lengths match
N = max(len(loom), len(tact))
if len(loom) != len(tact):
    print(f"\n=== PADDING TO MATCH LENGTHS ===")
    print(f"Padding to length: {N} samples")
    pad = lambda a: np.pad(a, ((0, N-len(a)), (0,0)), mode='constant')
    loom, tact = pad(loom), pad(tact)

# 4) If the input files are mono, convert to stereo
if loom.shape[1] == 1:
    print("Converting looming mono to stereo")
    loom = np.repeat(loom, 2, axis=1)
if tact.shape[1] == 1:
    print("Converting tactile mono to stereo")
    tact = np.repeat(tact, 2, axis=1)

# 5) Combine the two stereo pairs into a 4-channel file
merged = np.column_stack((loom, tact)).astype(np.float32)
print(f"\nCombined file shape: {merged.shape} (should be Nx4)")

# 6) Write the combined file as 24-bit PCM
sf.write(out_path, merged, sr, subtype="PCM_24")
print(f"\n✅ Created 4-channel file: {out_path}")
print(f"   - Shape: {merged.shape}")
print(f"   - Sample rate: {sr} Hz")
print(f"   - Channels: {merged.shape[1]}")
print(f"   - Duration: {len(merged)/sr:.2f} seconds")

# Verify the output file
print("\n=== VERIFYING OUTPUT FILE ===")
verify_audio, verify_sr = sf.read(out_path, always_2d=True)
print(f"File has {verify_audio.shape[1]} channels at {verify_sr} Hz")
print(f"This file should be compatible with your audio interface")

=== CHECKING AUDIO DEVICES ===
Device 25: Output 1/2 (2- Komplete Audio 6 MK2)
  Default sample rate: 44100.0 Hz
Device 22: Output 3/4 (2- Komplete Audio 6 MK2)
  Default sample rate: 44100.0 Hz
Target sample rate: 44100 Hz

=== READING INPUT FILES ===
Input file sample rate: 48000 Hz
Looming file shape: (80412672, 2)
Tactile file shape: (80412672, 2)

=== RESAMPLING FROM 48000 Hz TO 44100 Hz ===
After resampling - Looming shape: (73879142, 2)
After resampling - Tactile shape: (73879142, 2)

Combined file shape: (73879142, 4) (should be Nx4)

✅ Created 4-channel file: C:\Users\cogpsy-vrlab\Desktop\audio sample\participant_00_design_4ch.wav
   - Shape: (73879142, 4)
   - Sample rate: 44100 Hz
   - Channels: 4
   - Duration: 1675.26 seconds

=== VERIFYING OUTPUT FILE ===
File has 4 channels at 44100 Hz
This file should be compatible with your audio interface


In [ ]:
# Cell 2: Play the 4-channel WAV file through separate outputs
import sounddevice as sd
import soundfile as sf
import time
import pathlib

# Path to the 4-channel file
out_dir = r"C:\Users\cogpsy-vrlab\Desktop\audio sample"
out_file = "participant_00_design_4ch.wav"
wav_path = pathlib.Path(out_dir) / out_file

# Load the 4-channel file
audio, sr = sf.read(wav_path, always_2d=True)
print(f"Loaded {wav_path.name}")
print(f"Shape: {audio.shape}")
print(f"Sample rate: {sr} Hz")
print(f"Channels: {audio.shape[1]}")

# Extract channel pairs
channels_12 = audio[:, 0:2]  # First two channels (looming)
channels_34 = audio[:, 2:4]  # Last two channels (tactile)

# Select output devices for each stereo pair
output_12_id = 25  # Output 1/2 (2- Komplete Audio 6 MK2)
output_34_id = 22  # Output 3/4 (2- Komplete Audio 6 MK2)

print(f"Playing channels 1-2 on device {output_12_id}: {sd.query_devices(output_12_id)['name']}")
print(f"Playing channels 3-4 on device {output_34_id}: {sd.query_devices(output_34_id)['name']}")

# Function to play with manual buffer writing
def play_synchronized():
    print("\nPlaying synchronized audio... (Press Ctrl+C to stop)")
    try:
        # Create streams with no callback
        stream_12 = sd.OutputStream(
            samplerate=sr,
            device=output_12_id,
            channels=2,
            blocksize=1024
        )
        
        stream_34 = sd.OutputStream(
            samplerate=sr,
            device=output_34_id,
            channels=2,
            blocksize=1024
        )
        
        # Start both streams
        stream_12.start()
        stream_34.start()
        
        # Write data in chunks for synchronized playback
        chunk_size = 1024
        total_chunks = len(channels_12) // chunk_size
        
        for i in range(total_chunks):
            # Get chunk of data
            start = i * chunk_size
            end = min(start + chunk_size, len(channels_12))
            
            # Write to both streams
            stream_12.write(channels_12[start:end])
            stream_34.write(channels_34[start:end])
            
            # Display progress occasionally
            if i % 1000 == 0:
                print(f"Progress: {i/total_chunks*100:.1f}%")
        
        # Clean up streams
        stream_12.stop()
        stream_34.stop()
        stream_12.close()
        stream_34.close()
        
        print("Playback complete.")
        
    except KeyboardInterrupt:
        print("\nPlayback stopped by user.")
        try:
            stream_12.stop()
            stream_34.stop()
            stream_12.close()
            stream_34.close()
        except:
            pass
        
    except Exception as e:
        print(f"\nError during playback: {e}")
        try:
            if 'stream_12' in locals() and stream_12:
                stream_12.close()
            if 'stream_34' in locals() and stream_34:
                stream_34.close()
        except:
            pass
        
        print("\nTrying sequential playback as fallback...")
        play_sequential()

# Function to play sequentially (fallback)
def play_sequential():
    print("\nPlaying channels 1-2...")
    sd.play(channels_12, sr, device=output_12_id)
    sd.wait()
    
    print("Playing channels 3-4...")
    sd.play(channels_34, sr, device=output_34_id)
    sd.wait()
    
    print("Sequential playback complete.")

# Start playback
play_synchronized()

Loaded participant_00_design_4ch.wav
Shape: (73879142, 4)
Sample rate: 44100 Hz
Channels: 4
Playing channels 1-2 on device 25: Output 1/2 (2- Komplete Audio 6 MK2)
Playing channels 3-4 on device 22: Output 3/4 (2- Komplete Audio 6 MK2)

Playing synchronized audio... (Press Ctrl+C to stop)

Error during playback: dtype mismatch: 'float64' vs 'float32'

Trying sequential playback as fallback...

Playing channels 1-2...


In [1]:
# Cell 2: Diagnostic test with channel isolation
import sounddevice as sd
import soundfile as sf
import numpy as np
import time
import pathlib

# Path to the 4-channel file
out_dir = r"C:\Users\cogpsy-vrlab\Desktop\audio sample"
out_file = "participant_00_design_4ch.wav"
wav_path = pathlib.Path(out_dir) / out_file

# Load the 4-channel file
audio, sr = sf.read(wav_path, always_2d=True)
print(f"Loaded {wav_path.name}")
print(f"Shape: {audio.shape}")
print(f"Sample rate: {sr} Hz")
print(f"Channels: {audio.shape[1]}")

# Select output devices
output_12_id = 25  # Output 1/2 (2- Komplete Audio 6 MK2)
output_34_id = 22  # Output 3/4 (2- Komplete Audio 6 MK2)

print(f"Device 1-2: {sd.query_devices(output_12_id)['name']}")
print(f"Device 3-4: {sd.query_devices(output_34_id)['name']}")

# Extract and isolate channel pairs
if audio.shape[1] >= 4:
    # For diagnostic purposes, we'll extract specific channels and zero out others
    
    # Create isolated versions for testing
    channels_12_only = audio[:, 0:2].copy()  # First two channels only
    channels_34_only = audio[:, 2:4].copy()  # Last two channels only
    channels_12_zero = np.zeros_like(audio[:, 0:2])  # Silence for channels 1-2
    
    print("\n=== DIAGNOSTIC CHANNEL INFO ===")
    print(f"Channels 1-2 max value: {np.max(np.abs(channels_12_only))}")
    print(f"Channels 3-4 max value: {np.max(np.abs(channels_34_only))}")
else:
    print("Error: File doesn't have enough channels")
    
# Function to test one output at a time
def test_individual_outputs():
    print("\n=== TESTING INDIVIDUAL OUTPUTS ===")
    print("This will play looming on Output 1/2 only, then tactile on Output 3/4 only")
    
    # Play test tone on device 1-2 only
    print("\nPlaying LOOMING on Output 1/2 only... (3 seconds)")
    tone_12 = np.ones_like(channels_12_only[0:sr*3])  # 3 seconds of tone
    sd.play(tone_12 * 0.1, sr, device=output_12_id)  # Quiet tone
    sd.wait()
    
    time.sleep(1)  # Pause between tests
    
    # Play test tone on device 3-4 only
    print("Playing TACTILE on Output 3/4 only... (3 seconds)")
    tone_34 = np.ones_like(channels_34_only[0:sr*3])  # 3 seconds of tone
    sd.play(tone_34 * 0.1, sr, device=output_34_id)  # Quiet tone
    sd.wait()
    
    print("Test complete. Did you hear each sound only on its intended output?")

# Function to play short segments with clear identification
def play_diagnostic_segments():
    segment_length = 5 * sr  # 5 seconds
    
    if len(audio) < segment_length:
        segment_length = len(audio)
    
    print("\n=== PLAYING SHORT SEGMENTS WITH CHANNEL ISOLATION ===")
    
    # Play looming only on Output 1/2 (with silence on 3/4)
    print("Playing LOOMING ONLY on Output 1/2...")
    
    # Create isolated streams
    looming_segment = channels_12_only[0:segment_length]
    
    # Play looming through Output 1/2
    stream_12 = sd.OutputStream(samplerate=sr, device=output_12_id, channels=2)
    stream_12.start()
    stream_12.write(looming_segment)
    stream_12.stop()
    stream_12.close()
    
    time.sleep(1)  # Pause between tests
    
    # Play tactile only on Output 3/4 (with silence on 1/2)
    print("Playing TACTILE ONLY on Output 3/4...")
    
    # Create tactile segment
    tactile_segment = channels_34_only[0:segment_length]
    
    # Play tactile through Output 3/4
    stream_34 = sd.OutputStream(samplerate=sr, device=output_34_id, channels=2)
    stream_34.start()
    stream_34.write(tactile_segment)
    stream_34.stop()
    stream_34.close()
    
    print("Diagnostic complete. Did each sound play only on its respective output?")

# Test each output separately
test_individual_outputs()

# Then test with actual audio segments
play_diagnostic_segments()

print("\nIf either test failed, check your Komplete Audio 6 MK2 control panel settings.")
print("You may need to ensure that 'Direct Monitoring' is disabled or properly configured.")
print("Also check Windows sound settings for any duplicate routing.")

Loaded participant_00_design_4ch.wav
Shape: (73879142, 4)
Sample rate: 44100 Hz
Channels: 4
Device 1-2: Output 1/2 (2- Komplete Audio 6 MK2)
Device 3-4: Output 3/4 (2- Komplete Audio 6 MK2)

=== DIAGNOSTIC CHANNEL INFO ===
Channels 1-2 max value: 0.9350066184997559
Channels 3-4 max value: 0.95001220703125

=== TESTING INDIVIDUAL OUTPUTS ===
This will play looming on Output 1/2 only, then tactile on Output 3/4 only

Playing LOOMING on Output 1/2 only... (3 seconds)
Playing TACTILE on Output 3/4 only... (3 seconds)
Test complete. Did you hear each sound only on its intended output?

=== PLAYING SHORT SEGMENTS WITH CHANNEL ISOLATION ===
Playing LOOMING ONLY on Output 1/2...


TypeError: dtype mismatch: 'float64' vs 'float32'

In [9]:
import sounddevice as sd

print("=== Output-capable PortAudio devices ===")
for i, d in enumerate(sd.query_devices()):
    if d['max_output_channels'] > 0:
        name = d['name']
        host = sd.query_hostapis(d['hostapi'])['name']
        print(f"{i:2d}: {name}  [hostapi: {host}]  outs={d['max_output_channels']}")


=== Output-capable PortAudio devices ===
 5: Microsoft Sound Mapper - Output  [hostapi: MME]  outs=2
 6: Output 3/4 (2- Komplete Audio 6  [hostapi: MME]  outs=2
 7: SPDIF Output (2- Komplete Audio  [hostapi: MME]  outs=2
 8: Speakers (Realtek(R) Audio)  [hostapi: MME]  outs=2
 9: Output 1/2 (2- Komplete Audio 6  [hostapi: MME]  outs=2
10: DELL U2722D (HD Audio Driver fo  [hostapi: MME]  outs=2
16: Primary Sound Driver  [hostapi: Windows DirectSound]  outs=2
17: Output 3/4 (2- Komplete Audio 6 MK2)  [hostapi: Windows DirectSound]  outs=2
18: SPDIF Output (2- Komplete Audio 6 MK2)  [hostapi: Windows DirectSound]  outs=2
19: Speakers (Realtek(R) Audio)  [hostapi: Windows DirectSound]  outs=2
20: Output 1/2 (2- Komplete Audio 6 MK2)  [hostapi: Windows DirectSound]  outs=2
21: DELL U2722D (HD Audio Driver for Display Audio)  [hostapi: Windows DirectSound]  outs=2
22: Output 3/4 (2- Komplete Audio 6 MK2)  [hostapi: Windows WASAPI]  outs=2
23: SPDIF Output (2- Komplete Audio 6 MK2)  [hostapi:

In [3]:
!pip install --upgrade --force-reinstall sounddevice


  Using cached sounddevice-0.5.1-py3-none-win_amd64.whl.metadata (1.4 kB)
Using cached sounddevice-0.5.1-py3-none-win_amd64.whl (363 kB)
  Attempting uninstall: pycparser
    Found existing installation: pycparser 2.21
    Uninstalling pycparser-2.21:
      Successfully uninstalled pycparser-2.21
  Attempting uninstall: CFFI
    Found existing installation: cffi 1.17.1
    Uninstalling cffi-1.17.1:
      Successfully uninstalled cffi-1.17.1
  Attempting uninstall: sounddevice
    Found existing installation: sounddevice 0.5.1
    Uninstalling sounddevice-0.5.1:
      Successfully uninstalled sounddevice-0.5.1


  You can safely remove it manually.
  You can safely remove it manually.


In [4]:
27: Komplete Audio 6 MK2 ASIO  [hostapi: ASIO]  outs=4


SyntaxError: illegal target for annotation (3695553299.py, line 1)

In [2]:
import sounddevice as sd
import soundfile as sf
import pathlib

# Path to file (keep your existing path)
out_dir = r"C:\Users\cogpsy-vrlab\Desktop\audio sample"
out_file = "participant_00_design_4ch.wav" 
wav_path = pathlib.Path(out_dir) / out_file

# Load the 4-channel file
audio, sr = sf.read(wav_path, always_2d=True)
print(f"Loaded {wav_path.name}")
print(f"Shape: {audio.shape}")
print(f"Sample rate: {sr} Hz")
print(f"Channels: {audio.shape[1]}")

# Important: Use the ASIO device ID (likely ID 42 from your list)
# This appears to be your ASIO device for Komplete Audio
asio_device_id = 42  # Output 1/2 (Output 1/2) - ASIO version

try:
    print(f"\nPlaying all channels through ASIO device {asio_device_id}")
    # Note mapping starts at 1, not 0
    sd.play(audio, sr, device=asio_device_id, mapping=[1, 2, 3, 4], blocking=True)
    print("Playback complete")
except Exception as e:
    print(f"Error with mapping method: {e}")
    
    # Alternative approach using a raw output stream
    try:
        print("\nTrying with raw output stream...")
        # Create single output stream with all 4 channels
        with sd.RawOutputStream(
            samplerate=sr,
            device=asio_device_id,
            channels=4,  # Four channels total
            dtype='float32'
        ) as stream:
            # Start stream
            stream.start()
            
            # Convert audio data to proper type if needed
            if audio.dtype != 'float32':
                import numpy as np
                audio_data = audio.astype(np.float32)
            else:
                audio_data = audio
                
            # Write data to stream
            stream.write(audio_data)
            stream.stop()
            
        print("Playback complete")
    except Exception as e:
        print(f"Error with raw stream method: {e}")


Loaded participant_00_design_4ch.wav
Shape: (73879142, 4)
Sample rate: 44100 Hz
Channels: 4

Playing all channels through ASIO device 42
Error with mapping method: Error opening OutputStream: Invalid number of channels [PaErrorCode -9998]

Trying with raw output stream...
Error with raw stream method: Error opening RawOutputStream: Invalid number of channels [PaErrorCode -9998]


In [3]:
import sounddevice as sd
import soundfile as sf
import threading
import numpy as np
import pathlib
import time
import sys

def debug_print(message, indent=0):
    """Print debug messages with timestamps and optional indentation"""
    timestamp = time.strftime("%H:%M:%S", time.localtime())
    indent_str = "  " * indent
    print(f"[{timestamp}] {indent_str}{message}")

def list_audio_devices():
    """Print detailed information about all available audio devices"""
    debug_print("LISTING ALL AUDIO DEVICES:")
    debug_print("-" * 60, 1)
    
    devices = sd.query_devices()
    
    # First print a summary table
    debug_print("DEVICE SUMMARY:", 1)
    debug_print("ID  | Type       | Name                           | In/Out Channels", 1)
    debug_print("-" * 75, 1)
    
    for i, device in enumerate(devices):
        device_type = "Input/Output" if device['max_input_channels'] > 0 and device['max_output_channels'] > 0 else \
                     "Input Only" if device['max_input_channels'] > 0 else "Output Only"
        name = device['name'][:28] + "..." if len(device['name']) > 30 else device['name'].ljust(30)
        channels = f"{device['max_input_channels']} in / {device['max_output_channels']} out"
        debug_print(f"{i:<3} | {device_type:<10} | {name} | {channels}", 1)
    
    # Then print detailed information for each device
    debug_print("\nDETAILED DEVICE INFORMATION:", 1)
    for i, device in enumerate(devices):
        debug_print(f"Device {i}: {device['name']}", 1)
        debug_print(f"  Host API: {device['hostapi']}", 1)
        debug_print(f"  Channels: {device['max_input_channels']} inputs, {device['max_output_channels']} outputs", 1)
        if device['max_output_channels'] > 0:
            debug_print(f"  Default sample rate: {device['default_samplerate']} Hz", 1)
            debug_print(f"  Output latency: {device['default_low_output_latency']*1000:.2f}ms (low), {device['default_high_output_latency']*1000:.2f}ms (high)", 1)
        debug_print("", 1)  # Empty line for readability
    
    return devices

def find_komplete_audio_devices(devices):
    """Find devices that match Komplete Audio 6 and return likely output pairs"""
    debug_print("IDENTIFYING KOMPLETE AUDIO 6 DEVICES:")
    
    komplete_devices = []
    for i, device in enumerate(devices):
        if "komplete audio" in device['name'].lower() and device['max_output_channels'] > 0:
            komplete_devices.append((i, device))
    
    if not komplete_devices:
        debug_print("No Komplete Audio devices found!", 1)
        return None, None
    
    # Try to identify 1/2 and 3/4 output pairs
    output_12_id = None
    output_34_id = None
    
    debug_print(f"Found {len(komplete_devices)} Komplete Audio output devices:", 1)
    for idx, device in komplete_devices:
        debug_print(f"ID {idx}: {device['name']} ({device['max_output_channels']} channels)", 1)
        
        # Look for devices that might be the main output (1/2)
        if output_12_id is None and any(x in device['name'].lower() for x in ["1/2", "main", "1-2"]):
            output_12_id = idx
            debug_print(f"  → Identified as likely 1/2 outputs", 1)
        # Look for devices that might be the secondary output (3/4)
        elif output_34_id is None and any(x in device['name'].lower() for x in ["3/4", "3-4"]):
            output_34_id = idx
            debug_print(f"  → Identified as likely 3/4 outputs", 1)
    
    # If we couldn't identify by name, make educated guesses
    if output_12_id is None and len(komplete_devices) > 0:
        output_12_id = komplete_devices[0][0]
        debug_print(f"No explicit 1/2 output found, using first Komplete device (ID {output_12_id})", 1)
    
    if output_34_id is None and len(komplete_devices) > 1:
        output_34_id = komplete_devices[1][0]
        debug_print(f"No explicit 3/4 output found, using second Komplete device (ID {output_34_id})", 1)
    
    return output_12_id, output_34_id

def load_and_analyze_audio(file_path):
    """Load and analyze audio file, returning split channel arrays"""
    debug_print(f"LOADING AUDIO FILE: {file_path}")
    
    try:
        # Get file information
        file_info = sf.info(file_path)
        debug_print(f"File format: {file_info.format}", 1)
        debug_print(f"Sample rate: {file_info.samplerate} Hz", 1)
        debug_print(f"Channels: {file_info.channels}", 1)
        debug_print(f"Duration: {file_info.duration:.2f} seconds", 1)
        
        # Check if file has enough channels
        if file_info.channels < 4:
            debug_print(f"ERROR: File only has {file_info.channels} channels, need at least 4", 1)
            return None, None, None
        
        # Load the audio file
        debug_print("Reading audio data...", 1)
        audio, sr = sf.read(file_path, always_2d=True)
        debug_print(f"Successfully loaded: shape {audio.shape}, sample rate {sr} Hz", 1)
        
        # Split channels for different outputs
        channels_12 = audio[:, 0:2]  # First two channels
        channels_34 = audio[:, 2:4]  # Next two channels
        
        # Analyze audio levels
        debug_print("Channel analysis:", 1)
        for i in range(min(4, audio.shape[1])):
            channel_data = audio[:, i]
            peak = np.max(np.abs(channel_data))
            rms = np.sqrt(np.mean(channel_data**2))
            debug_print(f"  Channel {i+1}: Peak level = {20*np.log10(peak+1e-10):.1f} dB, RMS = {20*np.log10(rms+1e-10):.1f} dB", 1)
        
        return audio, channels_12, channels_34, sr
    
    except Exception as e:
        debug_print(f"ERROR loading audio file: {str(e)}", 1)
        return None, None, None, None

def play_multichannel_audio(channels_12, channels_34, sr, output_12_id, output_34_id):
    """Play split channels to different outputs with synchronization"""
    debug_print("PREPARING SYNCHRONIZED PLAYBACK:")
    
    # Check device latency
    try:
        latency_12 = sd.query_devices(output_12_id)['default_low_output_latency']
        latency_34 = sd.query_devices(output_12_id)['default_low_output_latency']
        debug_print(f"Output 1/2 latency: {latency_12*1000:.2f}ms", 1)
        debug_print(f"Output 3/4 latency: {latency_34*1000:.2f}ms", 1)
        
        latency_diff = abs(latency_12 - latency_34)
        if latency_diff > 0.005:  # More than 5ms difference
            debug_print(f"WARNING: Output devices have different latencies ({latency_diff*1000:.2f}ms difference)", 1)
    except Exception as e:
        debug_print(f"Could not query latency info: {e}", 1)
    
    # Create synchronization event
    start_event = threading.Event()
    playback_finished = [False, False]  # Track when each thread finishes
    
    # Define callback functions for stream completion
    def stream_12_callback(outdata, frames, time, status):
        if status:
            debug_print(f"Output 1/2 status: {status}", 1)
    
    def stream_34_callback(outdata, frames, time, status):
        if status:
            debug_print(f"Output 3/4 status: {status}", 1)
    
    # Thread functions
    def play_channels_12():
        debug_print(f"Thread for channels 1-2 ready (device {output_12_id})", 1)
        try:
            start_event.wait()  # Wait for sync signal
            debug_print("Starting playback on channels 1-2", 1)
            sd.play(channels_12, sr, device=output_12_id, blocking=True, callback=stream_12_callback)
            debug_print("Channels 1-2 playback complete", 1)
            playback_finished[0] = True
        except Exception as e:
            debug_print(f"ERROR in channels 1-2 playback: {str(e)}", 1)
    
    def play_channels_34():
        debug_print(f"Thread for channels 3-4 ready (device {output_34_id})", 1)
        try:
            start_event.wait()  # Wait for sync signal
            debug_print("Starting playback on channels 3-4", 1)
            sd.play(channels_34, sr, device=output_34_id, blocking=True, callback=stream_34_callback)
            debug_print("Channels 3-4 playback complete", 1)
            playback_finished[1] = True
        except Exception as e:
            debug_print(f"ERROR in channels 3-4 playback: {str(e)}", 1)
    
    # Create and start threads
    thread_12 = threading.Thread(target=play_channels_12)
    thread_34 = threading.Thread(target=play_channels_34)
    
    debug_print("Starting playback threads...", 1)
    thread_12.start()
    thread_34.start()
    
    # Give them a moment to initialize
    time.sleep(0.5)
    
    # Start playback simultaneously
    debug_print("Triggering synchronized playback...", 1)
    start_event.set()
    
    # Wait for both threads to complete (with timeout)
    max_wait_time = len(channels_12) / sr + 5  # Audio duration + 5 seconds
    start_time = time.time()
    
    debug_print(f"Waiting for playback to complete (timeout: {max_wait_time:.1f}s)...", 1)
    while not all(playback_finished) and (time.time() - start_time) < max_wait_time:
        time.sleep(0.5)
    
    if all(playback_finished):
        debug_print("All playback complete successfully", 1)
    else:
        debug_print("WARNING: Playback may not have completed properly", 1)
    
    return all(playback_finished)

def main():
    """Main function to run the program"""
    debug_print("MULTI-CHANNEL AUDIO PLAYBACK SCRIPT")
    debug_print("==================================")
    
    # List all audio devices
    devices = list_audio_devices()
    
    # Find Komplete Audio 6 devices
    output_12_id, output_34_id = find_komplete_audio_devices(devices)
    
    # Allow manual device selection if needed
    if output_12_id is None or output_34_id is None:
        debug_print("\nAUTOMATIC DEVICE DETECTION FAILED")
        debug_print("Please enter device IDs manually:")
        try:
            output_12_id = int(input("Enter device ID for outputs 1/2: "))
            output_34_id = int(input("Enter device ID for outputs 3/4: "))
        except ValueError:
            debug_print("Invalid input. Exiting.")
            return
    
    # Confirm selected devices
    debug_print("\nSELECTED OUTPUT DEVICES:")
    try:
        output_12_info = sd.query_devices(output_12_id)
        output_34_info = sd.query_devices(output_34_id)
        debug_print(f"Outputs 1/2: Device {output_12_id} - {output_12_info['name']}", 1)
        debug_print(f"Outputs 3/4: Device {output_34_id} - {output_34_info['name']}", 1)
    except Exception as e:
        debug_print(f"ERROR: Could not query device info: {e}", 1)
        return
    
    # Get audio file path
    debug_print("\nAUDIO FILE SELECTION:")
    if len(sys.argv) > 1:
        # Use command-line argument if provided
        wav_path = pathlib.Path(sys.argv[1])
        debug_print(f"Using file from command line: {wav_path}", 1)
    else:
        # Otherwise use default path
        out_dir = input("Enter directory path (or press Enter for default): ").strip()
        if not out_dir:
            out_dir = r"C:\Users\cogpsy-vrlab\Desktop\audio sample"
        
        out_file = input("Enter WAV filename (or press Enter for default): ").strip()
        if not out_file:
            out_file = "participant_00_design_4ch.wav"
        
        wav_path = pathlib.Path(out_dir) / out_file
        debug_print(f"Using file: {wav_path}", 1)
    
    # Verify file exists
    if not wav_path.exists():
        debug_print(f"ERROR: File not found: {wav_path}", 1)
        return
    
    # Load and analyze audio file
    audio, channels_12, channels_34, sr = load_and_analyze_audio(wav_path)
    if audio is None:
        debug_print("Failed to load audio. Exiting.")
        return
    
    # Play audio
    debug_print("\nSTARTING PLAYBACK:")
    success = play_multichannel_audio(channels_12, channels_34, sr, output_12_id, output_34_id)
    
    if success:
        debug_print("\nPLAYBACK COMPLETED SUCCESSFULLY")
    else:
        debug_print("\nPLAYBACK MAY HAVE ENCOUNTERED ISSUES")
    
    debug_print("Script execution complete.")

if __name__ == "__main__":
    main()

[00:57:14] MULTI-CHANNEL AUDIO PLAYBACK SCRIPT
[00:57:14] ==================================
[00:57:14] LISTING ALL AUDIO DEVICES:
[00:57:14]   ------------------------------------------------------------
[00:57:14]   DEVICE SUMMARY:
[00:57:14]   ID  | Type       | Name                           | In/Out Channels
[00:57:14]   ---------------------------------------------------------------------------
[00:57:14]   0   | Input Only | Microsoft Sound Mapper - Input | 2 in / 0 out
[00:57:14]   1   | Input Only | Input 1/2 (2- Komplete Audio... | 2 in / 0 out
[00:57:14]   2   | Input Only | Microphone Array (Intel® Sma... | 4 in / 0 out
[00:57:14]   3   | Input Only | Input 3/4 (2- Komplete Audio... | 2 in / 0 out
[00:57:14]   4   | Input Only | SPDIF Input (2- Komplete Aud... | 2 in / 0 out
[00:57:14]   5   | Output Only | Microsoft Sound Mapper - Out... | 0 in / 2 out
[00:57:14]   6   | Output Only | Output 1/2 (2- Komplete Audi... | 0 in / 2 out
[00:57:14]   7   | Output Only | Output 3/

In [9]:
import sounddevice as sd
import soundfile as sf
import threading
import numpy as np
import time

def debug_print(message, indent=0):
    """Print debug messages with timestamps and optional indentation"""
    timestamp = time.strftime("%H:%M:%S", time.localtime())
    indent_str = "  " * indent
    print(f"[{timestamp}] {indent_str}{message}")

def simple_resample(audio, orig_sr, target_sr):
    """Simple resampling function using linear interpolation"""
    debug_print(f"Resampling audio from {orig_sr}Hz to {target_sr}Hz...", 1)
    
    # Calculate the resampling ratio and new length
    ratio = target_sr / orig_sr
    new_length = int(len(audio) * ratio)
    
    # Create time arrays for interpolation
    orig_time = np.arange(len(audio)) / orig_sr
    new_time = np.arange(new_length) / target_sr
    
    # Create output array - explicitly use float32 for compatibility
    resampled = np.zeros((new_length, audio.shape[1]), dtype=np.float32)
    
    # Resample each channel separately using linear interpolation
    for channel in range(audio.shape[1]):
        resampled[:, channel] = np.interp(new_time, orig_time, audio[:, channel])
    
    debug_print(f"Resampling complete. Original samples: {len(audio)}, New samples: {len(resampled)}", 1)
    debug_print(f"Data type: {resampled.dtype}", 1)
    return resampled

def play_stereo_to_separate_outputs(file_path, output_12_id, output_34_id):
    """Load stereo file and route left to output 1/2 and right to output 3/4"""
    debug_print("ANALYZING AUDIO FILE AND DEVICES:")
    
    try:
        # Step 1: Check device sample rates
        device_12_info = sd.query_devices(output_12_id)
        device_34_info = sd.query_devices(output_34_id)
        
        device_sr = int(device_12_info['default_samplerate'])
        debug_print(f"Device sample rate: {device_sr}Hz", 1)
        
        # Step 2: Load and analyze the audio file
        debug_print(f"Loading file: {file_path}", 1)
        audio, file_sr = sf.read(file_path, always_2d=True, dtype='float32')
        debug_print(f"File loaded: {len(audio)} samples, {file_sr}Hz, {audio.shape[1]} channels, {audio.dtype}", 1)
        
        # Step 3: Resample if necessary
        if file_sr != device_sr:
            debug_print(f"Sample rate mismatch: file={file_sr}Hz, device={device_sr}Hz", 1)
            audio = simple_resample(audio, file_sr, device_sr)
        
        # Step 4: Split the channels and ensure float32 format
        left_channel = audio[:, 0].copy().astype(np.float32)
        right_channel = audio[:, 1].copy().astype(np.float32)
        
        debug_print(f"Left channel: {len(left_channel)} samples, {left_channel.dtype}", 1)
        debug_print(f"Right channel: {len(right_channel)} samples, {right_channel.dtype}", 1)
        
        # Step 5: Prepare audio for each output
        # For Output 1/2: Left channel duplicated to both stereo channels
        left_output = np.column_stack((left_channel, left_channel))
        
        # For Output 3/4: Right channel duplicated to both stereo channels
        right_output = np.column_stack((right_channel, right_channel))
        
        # Ensure data is float32
        left_output = left_output.astype(np.float32)
        right_output = right_output.astype(np.float32)
        
        debug_print("READY FOR PLAYBACK:", 1)
        debug_print(f"Left output shape: {left_output.shape}, dtype: {left_output.dtype}", 1)
        debug_print(f"Right output shape: {right_output.shape}, dtype: {right_output.dtype}", 1)
        
        # Step 6: Synchronized playback
        debug_print("\nINITIATING SYNCHRONIZED PLAYBACK:")
        
        # Use a countdown for better synchronization perception
        for i in range(3, 0, -1):
            debug_print(f"Starting in {i}...", 1)
            time.sleep(1)
        
        debug_print("⏯️ STARTING PLAYBACK NOW!", 1)
        
        # Try the simplest approach first - direct streams with explicit settings
        try:
            # Create a shared event for synchronization
            start_event = threading.Event()
            
            # Setup block size for streams
            blocksize = 1024  # Standard block size for low latency
            
            # Create streams but don't start them yet
            streams = []
            
            # Data buffers for callback functions
            buffer_left = {'data': left_output, 'position': 0, 'finished': False}
            buffer_right = {'data': right_output, 'position': 0, 'finished': False}
            
            # Create callback functions
            def callback_left(outdata, frames, time, status):
                if status:
                    debug_print(f"Left channel status: {status}", 2)
                
                pos = buffer_left['position']
                chunk = buffer_left['data'][pos:pos+frames]
                
                # If we have enough frames, write them
                if len(chunk) == frames:
                    outdata[:] = chunk
                    buffer_left['position'] += frames
                # If we have some frames but not enough, pad with zeros
                elif len(chunk) > 0:
                    outdata[:len(chunk)] = chunk
                    outdata[len(chunk):] = 0
                    buffer_left['position'] += len(chunk)
                    if not buffer_left['finished']:
                        buffer_left['finished'] = True
                        debug_print("Left channel playback complete", 1)
                # If no frames left, just output zeros
                else:
                    outdata[:] = 0
            
            def callback_right(outdata, frames, time, status):
                if status:
                    debug_print(f"Right channel status: {status}", 2)
                
                pos = buffer_right['position']
                chunk = buffer_right['data'][pos:pos+frames]
                
                # If we have enough frames, write them
                if len(chunk) == frames:
                    outdata[:] = chunk
                    buffer_right['position'] += frames
                # If we have some frames but not enough, pad with zeros
                elif len(chunk) > 0:
                    outdata[:len(chunk)] = chunk
                    outdata[len(chunk):] = 0
                    buffer_right['position'] += len(chunk)
                    if not buffer_right['finished']:
                        buffer_right['finished'] = True
                        debug_print("Right channel playback complete", 1)
                # If no frames left, just output zeros
                else:
                    outdata[:] = 0
            
            # Create both streams
            stream_left = sd.OutputStream(
                samplerate=device_sr,
                blocksize=blocksize,
                device=output_12_id,
                channels=2,
                dtype='float32',
                callback=callback_left
            )
            
            stream_right = sd.OutputStream(
                samplerate=device_sr,
                blocksize=blocksize,
                device=output_34_id,
                channels=2,
                dtype='float32',
                callback=callback_right
            )
            
            # Open both streams
            stream_left.start()
            stream_right.start()
            
            debug_print("Playback started successfully!", 1)
            debug_print(f"Playing {len(audio)/device_sr:.1f} seconds of audio...", 1)
            
            # Let them run until completion
            try:
                # Periodically check if playback is complete
                total_duration = len(audio) / device_sr
                start_time = time.time()
                
                while time.time() - start_time < total_duration + 1:  # +1 second buffer
                    time.sleep(1)
                    elapsed = time.time() - start_time
                    percent = min(100, int(elapsed / total_duration * 100))
                    debug_print(f"Playback progress: {percent}% ({elapsed:.1f}s / {total_duration:.1f}s)", 1)
                    
                    # Check if both channels are finished
                    if buffer_left['finished'] and buffer_right['finished']:
                        break
                
                debug_print("Playback complete", 1)
            except KeyboardInterrupt:
                debug_print("Playback interrupted by user", 1)
            finally:
                # Clean up streams
                stream_left.stop()
                stream_right.stop()
                stream_left.close()
                stream_right.close()
            
            return True
            
        except Exception as e:
            debug_print(f"Error during playback: {e}", 1)
            return False
    
    except Exception as e:
        debug_print(f"ERROR: {e}", 1)
        return False

def main():
    """Main function to run the script"""
    debug_print("STEREO TO DUAL MONO SYNCHRONIZED PLAYBACK")
    debug_print("========================================")
    
    # Explicitly use the WASAPI devices with lowest latency
    output_12_id = 21  # Output 1/2
    output_34_id = 20  # Output 3/4
    
    debug_print("USING DEVICES:")
    debug_print(f"Left channel → Device {output_12_id} (Output 1/2)", 1)
    debug_print(f"Right channel → Device {output_34_id} (Output 3/4)", 1)
    
    # File path for your 2-channel WAV
    file_path = r"C:\Users\cogpsy-vrlab\Desktop\audio sample\participant_00_design_2ch.wav"
    debug_print(f"AUDIO FILE: {file_path}")
    
    # Play the audio
    success = play_stereo_to_separate_outputs(file_path, output_12_id, output_34_id)
    
    if success:
        debug_print("✓ PLAYBACK COMPLETED SUCCESSFULLY")
    else:
        debug_print("⚠️ PLAYBACK ENCOUNTERED ISSUES")

if __name__ == "__main__":
    main()

[02:46:56] STEREO TO DUAL MONO SYNCHRONIZED PLAYBACK
[02:46:56] ========================================
[02:46:56] USING DEVICES:
[02:46:56]   Left channel → Device 21 (Output 1/2)
[02:46:56]   Right channel → Device 20 (Output 3/4)
[02:46:56] AUDIO FILE: C:\Users\cogpsy-vrlab\Desktop\audio sample\participant_00_design_2ch.wav
[02:46:56] ANALYZING AUDIO FILE AND DEVICES:
[02:46:56]   Device sample rate: 44100Hz
[02:46:56]   Loading file: C:\Users\cogpsy-vrlab\Desktop\audio sample\participant_00_design_2ch.wav
[02:46:57]   File loaded: 80412672 samples, 48000Hz, 2 channels, float32
[02:46:57]   Sample rate mismatch: file=48000Hz, device=44100Hz
[02:46:57]   Resampling audio from 48000Hz to 44100Hz...
[02:46:59]   Resampling complete. Original samples: 80412672, New samples: 73879142
[02:46:59]   Data type: float32
[02:46:59]   Left channel: 73879142 samples, float32
[02:46:59]   Right channel: 73879142 samples, float32
[02:47:00]   READY FOR PLAYBACK:
[02:47:00]   Left output shape: (7

In [10]:
import sounddevice as sd
import soundfile as sf
import threading
import numpy as np
import time
import os

def debug_print(message, indent=0):
    """Print debug messages with timestamps and optional indentation"""
    timestamp = time.strftime("%H:%M:%S", time.localtime())
    indent_str = "  " * indent
    print(f"[{timestamp}] {indent_str}{message}")

def chunked_resample(audio_file, orig_sr, target_sr, chunk_size=1024*1024):
    """Resample audio file in chunks to preserve memory"""
    # Get file info
    info = sf.info(audio_file)
    total_frames = info.frames
    channels = info.channels
    
    # Calculate output size
    ratio = target_sr / orig_sr
    output_frames = int(total_frames * ratio)
    
    debug_print(f"Chunked resampling: {total_frames} frames @ {orig_sr}Hz → {output_frames} frames @ {target_sr}Hz", 1)
    
    # Pre-allocate output array (float32 for device compatibility)
    resampled = np.zeros((output_frames, channels), dtype=np.float32)
    
    # Process in chunks to save memory
    chunks_processed = 0
    frames_processed = 0
    output_pos = 0
    
    with sf.SoundFile(audio_file) as f:
        while frames_processed < total_frames:
            # Read a chunk (ensure float32 format)
            chunk_frames = min(chunk_size, total_frames - frames_processed)
            chunk = f.read(chunk_frames, dtype='float32')
            
            # Process this chunk
            chunk_time = np.arange(frames_processed, frames_processed + len(chunk)) / orig_sr
            
            # Calculate corresponding output chunk
            out_start = int(frames_processed * ratio)
            out_end = int((frames_processed + len(chunk)) * ratio)
            out_frames = out_end - out_start
            
            # Create time arrays for interpolation
            new_time = np.arange(out_start, out_end) / target_sr
            
            # Resample each channel
            for ch in range(channels):
                resampled[out_start:out_end, ch] = np.interp(new_time, chunk_time, chunk[:, ch])
            
            # Update positions
            frames_processed += len(chunk)
            chunks_processed += 1
            
            # Progress update
            if chunks_processed % 10 == 0:
                percent = int(frames_processed / total_frames * 100)
                debug_print(f"Resampling progress: {percent}%", 2)
    
    debug_print(f"Resampling complete: {chunks_processed} chunks processed", 1)
    return resampled, target_sr

class SynchronizedPlayer:
    """Class to handle perfectly synchronized playback across multiple audio devices"""
    
    def __init__(self):
        self.playing = False
        self.streams = []
        self.buffers = []
        self.blocksize = 256  # Smaller blocksize for better synchronization
        self.latency = 'low'  # Use low latency setting
    
    def prepare_playback(self, channels, sample_rate, device_ids):
        """Set up streams for synchronized playback"""
        self.sample_rate = sample_rate
        self.device_ids = device_ids
        self.channels = channels
        self.buffers = []
        self.streams = []
        
        # Get device info and confirm compatibility
        for i, dev_id in enumerate(device_ids):
            try:
                device_info = sd.query_devices(dev_id)
                debug_print(f"Device {dev_id}: {device_info['name']}", 1)
                
                if device_info['default_samplerate'] != sample_rate:
                    debug_print(f"WARNING: Device {dev_id} default rate ({device_info['default_samplerate']}Hz) differs from target rate ({sample_rate}Hz)", 1)
            except Exception as e:
                debug_print(f"Error checking device {dev_id}: {e}", 1)
                return False
        
        # Create buffers for each channel
        for ch in channels:
            self.buffers.append({
                'data': ch.astype(np.float32),
                'position': 0,
                'finished': False
            })
        
        # Create callbacks for each device
        callbacks = []
        for i in range(len(device_ids)):
            def make_callback(idx):
                def callback(outdata, frames, time, status):
                    if status:
                        debug_print(f"Stream {idx} status: {status}", 2)
                    
                    buffer = self.buffers[idx]
                    pos = buffer['position']
                    chunk = buffer['data'][pos:pos+frames]
                    
                    if len(chunk) == frames:
                        outdata[:] = chunk
                        buffer['position'] += frames
                    elif len(chunk) > 0:
                        outdata[:len(chunk)] = chunk
                        outdata[len(chunk):] = 0
                        buffer['position'] += len(chunk)
                        if not buffer['finished']:
                            buffer['finished'] = True
                            debug_print(f"Stream {idx} playback complete", 1)
                    else:
                        outdata.fill(0)
                return callback
            callbacks.append(make_callback(i))
        
        # Create streams but don't start them yet
        for i, dev_id in enumerate(device_ids):
            try:
                stream = sd.OutputStream(
                    samplerate=sample_rate,
                    blocksize=self.blocksize,
                    device=dev_id,
                    channels=2,  # Always use stereo outputs
                    dtype='float32',  # Must use float32 for WASAPI
                    callback=callbacks[i],
                    latency=self.latency
                )
                self.streams.append(stream)
                debug_print(f"Stream {i} created for device {dev_id}", 1)
            except Exception as e:
                debug_print(f"Error creating stream for device {dev_id}: {e}", 1)
                self.cleanup()
                return False
        
        return True
    
    def start_playback(self):
        """Start synchronized playback across all devices"""
        if not self.streams:
            debug_print("No streams to start", 1)
            return False
        
        try:
            # Open all streams first without starting
            for i, stream in enumerate(self.streams):
                stream.start()
                debug_print(f"Stream {i} started", 2)
            
            self.playing = True
            self.start_time = time.time()
            debug_print("✓ All streams started successfully", 1)
            return True
        except Exception as e:
            debug_print(f"Error starting playback: {e}", 1)
            self.cleanup()
            return False
    
    def wait_for_completion(self):
        """Wait for all streams to complete playback"""
        if not self.playing:
            return False
        
        try:
            # Get duration of audio
            if self.buffers and 'data' in self.buffers[0]:
                duration = len(self.buffers[0]['data']) / self.sample_rate
            else:
                duration = 60  # Default timeout
            
            debug_print(f"Waiting for {duration:.1f}s of playback to complete...", 1)
            
            # Wait for playback to complete
            start_time = time.time()
            last_progress = -1
            
            while time.time() - start_time < duration + 2:  # Add 2s buffer
                # Check if all buffers are finished
                if all(buffer['finished'] for buffer in self.buffers):
                    debug_print("All streams completed playback", 1)
                    break
                
                # Show progress
                elapsed = time.time() - start_time
                progress = int((elapsed / duration) * 100)
                
                if progress != last_progress and progress % 5 == 0:
                    debug_print(f"Playback progress: {progress}% ({elapsed:.1f}s / {duration:.1f}s)", 1)
                    last_progress = progress
                
                time.sleep(0.25)  # Check more frequently for better progress updates
            
            return True
        
        except KeyboardInterrupt:
            debug_print("Playback interrupted by user", 1)
            return False
        finally:
            self.cleanup()
    
    def cleanup(self):
        """Clean up all streams"""
        if self.streams:
            for i, stream in enumerate(self.streams):
                try:
                    if stream.active:
                        stream.stop()
                    stream.close()
                    debug_print(f"Stream {i} closed", 2)
                except Exception as e:
                    debug_print(f"Error closing stream {i}: {e}", 2)
            
            self.streams = []
        
        self.playing = False
        debug_print("All streams cleaned up", 1)

def play_stereo_split(file_path, device_left, device_right):
    """Play stereo file split across two devices with perfect synchronization"""
    debug_print("PREPARING AUDIO FILE:")
    
    try:
        # Get device info
        device_left_info = sd.query_devices(device_left)
        device_right_info = sd.query_devices(device_right)
        
        # Get device sample rate (use the same for both)
        device_sr = int(device_left_info['default_samplerate'])
        debug_print(f"Device sample rate: {device_sr}Hz", 1)
        
        # Get file info first
        file_info = sf.info(file_path)
        file_sr = file_info.samplerate
        file_channels = file_info.channels
        duration = file_info.duration
        
        debug_print(f"File info: {file_sr}Hz, {file_channels} channels, {duration:.1f} seconds", 1)
        
        # Check if we need to resample
        if file_sr != device_sr:
            debug_print(f"Need to resample from {file_sr}Hz to {device_sr}Hz", 1)
            
            # Analyze the file size to determine if chunked resampling is needed
            file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
            
            if file_size_mb > 100:  # If file is larger than 100MB, use chunked resampling
                debug_print(f"Large file detected ({file_size_mb:.1f}MB), using chunked resampling", 1)
                audio, _ = chunked_resample(file_path, file_sr, device_sr)
            else:
                # For smaller files, load into memory and resample all at once
                debug_print(f"Loading file into memory ({file_size_mb:.1f}MB)", 1)
                audio, _ = sf.read(file_path, dtype='float32')
                
                # Simple resampling
                debug_print("Resampling audio...", 1)
                ratio = device_sr / file_sr
                new_length = int(len(audio) * ratio)
                
                # Create time arrays
                orig_time = np.arange(len(audio)) / file_sr
                new_time = np.arange(new_length) / device_sr
                
                # Create output array with float32 dtype
                resampled = np.zeros((new_length, file_channels), dtype=np.float32)
                
                # Resample each channel
                for ch in range(file_channels):
                    resampled[:, ch] = np.interp(new_time, orig_time, audio[:, ch])
                
                audio = resampled
        else:
            # No resampling needed, just load the file
            debug_print("Sample rates match, no resampling needed", 1)
            audio, _ = sf.read(file_path, dtype='float32')
        
        # Ensure we have a stereo file
        if audio.shape[1] != 2:
            debug_print(f"ERROR: File has {audio.shape[1]} channels, expected 2", 1)
            return False
        
        # Create dedicated stereo outputs for each device
        debug_print("Preparing channel routing...", 1)
        
        # For Output 1/2: Left channel duplicated to both stereo channels
        left_stereo = np.column_stack((audio[:, 0], audio[:, 0])).astype(np.float32)
        
        # For Output 3/4: Right channel duplicated to both stereo channels
        right_stereo = np.column_stack((audio[:, 1], audio[:, 1])).astype(np.float32)
        
        debug_print(f"Audio prepared: {len(audio)} samples, {audio.shape[1]} channels", 1)
        
        # Create player and prepare streams
        player = SynchronizedPlayer()
        
        debug_print("Setting up synchronized streams...", 1)
        if not player.prepare_playback([left_stereo, right_stereo], device_sr, [device_left, device_right]):
            debug_print("Failed to prepare playback", 1)
            return False
        
        # Start playback with countdown
        debug_print("\nINITIATING SYNCHRONIZED PLAYBACK:")
        for i in range(3, 0, -1):
            debug_print(f"Starting in {i}...", 1)
            time.sleep(1)
        
        debug_print("⏯️ STARTING PLAYBACK NOW!", 1)
        
        # Start playback
        if not player.start_playback():
            debug_print("Failed to start playback", 1)
            return False
        
        # Wait for completion
        success = player.wait_for_completion()
        
        if success:
            debug_print("✓ PLAYBACK COMPLETED SUCCESSFULLY")
            return True
        else:
            debug_print("⚠️ PLAYBACK ENCOUNTERED ISSUES")
            return False
        
    except Exception as e:
        debug_print(f"ERROR: {e}", 1)
        return False

def main():
    """Main function"""
    debug_print("OPTIMIZED SYNCHRONIZED STEREO PLAYBACK")
    debug_print("=====================================")
    
    # Explicitly use the WASAPI devices with lowest latency
    output_12_id = 21  # Output 1/2
    output_34_id = 20  # Output 3/4
    
    debug_print("USING DEVICES:")
    debug_print(f"Left channel → Device {output_12_id} (Output 1/2)", 1)
    debug_print(f"Right channel → Device {output_34_id} (Output 3/4)", 1)
    
    # File path for your 2-channel WAV
    file_path = r"C:\Users\cogpsy-vrlab\Desktop\audio sample\participant_00_design_2ch.wav"
    debug_print(f"AUDIO FILE: {file_path}")
    
    # Play the audio
    play_stereo_split(file_path, output_12_id, output_34_id)

if __name__ == "__main__":
    main()

[02:48:47] OPTIMIZED SYNCHRONIZED STEREO PLAYBACK
[02:48:47] =====================================
[02:48:47] USING DEVICES:
[02:48:47]   Left channel → Device 21 (Output 1/2)
[02:48:47]   Right channel → Device 20 (Output 3/4)
[02:48:47] AUDIO FILE: C:\Users\cogpsy-vrlab\Desktop\audio sample\participant_00_design_2ch.wav
[02:48:47] PREPARING AUDIO FILE:
[02:48:47]   Device sample rate: 44100Hz
[02:48:47]   File info: 48000Hz, 2 channels, 1675.3 seconds
[02:48:47]   Need to resample from 48000Hz to 44100Hz
[02:48:47]   Large file detected (460.1MB), using chunked resampling
[02:48:47]   Chunked resampling: 80412672 frames @ 48000Hz → 73879142 frames @ 44100Hz
[02:48:48]     Resampling progress: 13%
[02:48:48]     Resampling progress: 26%
[02:48:48]     Resampling progress: 39%
[02:48:49]     Resampling progress: 52%
[02:48:49]     Resampling progress: 65%
[02:48:49]     Resampling progress: 78%
[02:48:50]     Resampling progress: 91%
[02:48:50]   Resampling complete: 77 chunks processe

In [11]:
import sounddevice as sd
import soundfile as sf
import threading
import numpy as np
import time

def debug_print(message, indent=0):
    """Print debug messages with timestamps and optional indentation"""
    timestamp = time.strftime("%H:%M:%S", time.localtime())
    indent_str = "  " * indent
    print(f"[{timestamp}] {indent_str}{message}")

def simple_resample(audio, orig_sr, target_sr):
    """Simple resampling function using linear interpolation"""
    debug_print(f"Resampling audio from {orig_sr}Hz to {target_sr}Hz...", 1)
    
    # Calculate the resampling ratio and new length
    ratio = target_sr / orig_sr
    new_length = int(len(audio) * ratio)
    
    # Create time arrays for interpolation
    orig_time = np.arange(len(audio)) / orig_sr
    new_time = np.arange(new_length) / target_sr
    
    # Create output array - explicitly use float32 for compatibility
    resampled = np.zeros((new_length, audio.shape[1]), dtype=np.float32)
    
    # Resample each channel separately using linear interpolation
    for channel in range(audio.shape[1]):
        resampled[:, channel] = np.interp(new_time, orig_time, audio[:, channel])
    
    debug_print(f"Resampling complete. Original samples: {len(audio)}, New samples: {len(resampled)}", 1)
    debug_print(f"Data type: {resampled.dtype}", 1)
    return resampled

def play_stereo_to_separate_outputs(file_path, output_12_id, output_34_id):
    """Load stereo file and route left to output 1/2 and right to output 3/4"""
    debug_print("ANALYZING AUDIO FILE AND DEVICES:")
    
    try:
        # Step 1: Check device sample rates
        device_12_info = sd.query_devices(output_12_id)
        device_34_info = sd.query_devices(output_34_id)
        
        device_sr = int(device_12_info['default_samplerate'])
        debug_print(f"Device sample rate: {device_sr}Hz", 1)
        
        # Step 2: Load and analyze the audio file
        debug_print(f"Loading file: {file_path}", 1)
        audio, file_sr = sf.read(file_path, always_2d=True, dtype='float32')
        debug_print(f"File loaded: {len(audio)} samples, {file_sr}Hz, {audio.shape[1]} channels, {audio.dtype}", 1)
        
        # Step 3: Resample if necessary
        if file_sr != device_sr:
            debug_print(f"Sample rate mismatch: file={file_sr}Hz, device={device_sr}Hz", 1)
            audio = simple_resample(audio, file_sr, device_sr)
        
        # Step 4: Split the channels and ensure float32 format
        left_channel = audio[:, 0].copy().astype(np.float32)
        right_channel = audio[:, 1].copy().astype(np.float32)
        
        # Generate test signals to verify different audio in each channel
        # Create a test sine tone for the left channel (440 Hz = A4 note)
        debug_print("Creating test signals to verify different channels...", 1)
        test_duration = 3.0  # seconds
        test_samples = int(test_duration * device_sr)
        
        # Generate different test tones for each channel
        t = np.linspace(0, test_duration, test_samples, endpoint=False)
        test_left = 0.5 * np.sin(2 * np.pi * 440 * t).astype(np.float32)  # 440 Hz = A4
        test_right = 0.5 * np.sin(2 * np.pi * 880 * t).astype(np.float32)  # 880 Hz = A5 (one octave higher)
        
        # Insert test tones at the beginning of each channel
        if len(left_channel) > test_samples:
            # Fade in/out for the test tones
            fade_samples = int(0.1 * device_sr)  # 100ms fade
            fade_in = np.linspace(0, 1, fade_samples)
            fade_out = np.linspace(1, 0, fade_samples)
            
            # Apply fades
            test_left[:fade_samples] *= fade_in
            test_left[-fade_samples:] *= fade_out
            test_right[:fade_samples] *= fade_in
            test_right[-fade_samples:] *= fade_out
            
            # Insert the test tones
            left_channel[:test_samples] = test_left
            right_channel[:test_samples] = test_right
            
            debug_print(f"Added test tones: 440 Hz (A4) to left, 880 Hz (A5) to right", 1)
        
        debug_print(f"Left channel: {len(left_channel)} samples, {left_channel.dtype}", 1)
        debug_print(f"Right channel: {len(right_channel)} samples, {right_channel.dtype}", 1)
        
        # Make stereo silence for testing
        silence = np.zeros((int(device_sr * 0.5), 2), dtype=np.float32)  # 0.5 second of silence
        
        # Check for hardware direct monitoring
        debug_print("⚠️ IMPORTANT: If you hear the test tones on both outputs, disable Direct Monitoring", 1)
        debug_print("on your Komplete Audio 6 MK2 hardware (turn the DIRECT MONITORING knob fully counter-clockwise)", 1)
        
        # Step 5: Prepare audio for each output with zero in unused channel
        # For Output 1/2: Left channel to left, silence to right
        left_output = np.column_stack((left_channel, np.zeros_like(left_channel)))
        
        # For Output 3/4: Silence to left, right channel to right 
        right_output = np.column_stack((np.zeros_like(right_channel), right_channel))
        
        # Ensure data is float32
        left_output = left_output.astype(np.float32)
        right_output = right_output.astype(np.float32)
        
        debug_print("READY FOR PLAYBACK:", 1)
        debug_print(f"Left output shape: {left_output.shape}, dtype: {left_output.dtype}", 1)
        debug_print(f"Right output shape: {right_output.shape}, dtype: {right_output.dtype}", 1)
        
        # Step 6: Synchronized playback
        debug_print("\nINITIATING SYNCHRONIZED PLAYBACK:")
        
        # Use a countdown for better synchronization perception
        for i in range(3, 0, -1):
            debug_print(f"Starting in {i}...", 1)
            time.sleep(1)
        
        debug_print("⏯️ STARTING PLAYBACK NOW!", 1)
        
        # Try the simplest approach first - direct streams with explicit settings
        try:
            # Create a shared event for synchronization
            start_event = threading.Event()
            
            # Setup block size for streams - smaller for better timing
            blocksize = 256  # Smaller blocksize for lower latency
            
            # Create streams but don't start them yet
            streams = []
            
            # Data buffers for callback functions
            buffer_left = {'data': left_output, 'position': 0, 'finished': False}
            buffer_right = {'data': right_output, 'position': 0, 'finished': False}
            
            # Create callback functions
            def callback_left(outdata, frames, time, status):
                if status:
                    debug_print(f"Left channel status: {status}", 2)
                
                pos = buffer_left['position']
                chunk = buffer_left['data'][pos:pos+frames]
                
                # If we have enough frames, write them
                if len(chunk) == frames:
                    outdata[:] = chunk
                    buffer_left['position'] += frames
                # If we have some frames but not enough, pad with zeros
                elif len(chunk) > 0:
                    outdata[:len(chunk)] = chunk
                    outdata[len(chunk):] = 0
                    buffer_left['position'] += len(chunk)
                    if not buffer_left['finished']:
                        buffer_left['finished'] = True
                        debug_print("Left channel playback complete", 1)
                # If no frames left, just output zeros
                else:
                    outdata[:] = 0
            
            def callback_right(outdata, frames, time, status):
                if status:
                    debug_print(f"Right channel status: {status}", 2)
                
                pos = buffer_right['position']
                chunk = buffer_right['data'][pos:pos+frames]
                
                # If we have enough frames, write them
                if len(chunk) == frames:
                    outdata[:] = chunk
                    buffer_right['position'] += frames
                # If we have some frames but not enough, pad with zeros
                elif len(chunk) > 0:
                    outdata[:len(chunk)] = chunk
                    outdata[len(chunk):] = 0
                    buffer_right['position'] += len(chunk)
                    if not buffer_right['finished']:
                        buffer_right['finished'] = True
                        debug_print("Right channel playback complete", 1)
                # If no frames left, just output zeros
                else:
                    outdata[:] = 0
            
            # Create both streams
            stream_left = sd.OutputStream(
                samplerate=device_sr,
                blocksize=blocksize,
                device=output_12_id,
                channels=2,
                dtype='float32',
                callback=callback_left,
                latency='low'  # Use low latency mode
            )
            
            stream_right = sd.OutputStream(
                samplerate=device_sr,
                blocksize=blocksize,
                device=output_34_id,
                channels=2,
                dtype='float32',
                callback=callback_right,
                latency='low'  # Use low latency mode
            )
            
            # Open both streams
            stream_left.start()
            stream_right.start()
            
            debug_print("Playback started successfully!", 1)
            debug_print(f"Playing {len(audio)/device_sr:.1f} seconds of audio...", 1)
            debug_print("⚠️ First 3 seconds: Test tones (440Hz left, 880Hz right)", 1)
            
            # Let them run until completion
            try:
                # Periodically check if playback is complete
                total_duration = len(audio) / device_sr
                start_time = time.time()
                
                while time.time() - start_time < total_duration + 1:  # +1 second buffer
                    time.sleep(1)
                    elapsed = time.time() - start_time
                    percent = min(100, int(elapsed / total_duration * 100))
                    debug_print(f"Playback progress: {percent}% ({elapsed:.1f}s / {total_duration:.1f}s)", 1)
                    
                    # Check if both channels are finished
                    if buffer_left['finished'] and buffer_right['finished']:
                        break
                
                debug_print("Playback complete", 1)
            except KeyboardInterrupt:
                debug_print("Playback interrupted by user", 1)
            finally:
                # Clean up streams
                stream_left.stop()
                stream_right.stop()
                stream_left.close()
                stream_right.close()
            
            return True
            
        except Exception as e:
            debug_print(f"Error during playback: {e}", 1)
            return False
    
    except Exception as e:
        debug_print(f"ERROR: {e}", 1)
        return False

def main():
    """Main function to run the script"""
    debug_print("STEREO TO DUAL MONO SYNCHRONIZED PLAYBACK")
    debug_print("========================================")
    
    # Explicitly use the WASAPI devices with lowest latency
    output_12_id = 21  # Output 1/2
    output_34_id = 20  # Output 3/4
    
    debug_print("USING DEVICES:")
    debug_print(f"Left channel → Device {output_12_id} (Output 1/2)", 1)
    debug_print(f"Right channel → Device {output_34_id} (Output 3/4)", 1)
    
    # File path for your 2-channel WAV
    file_path = r"C:\Users\cogpsy-vrlab\Desktop\audio sample\participant_00_design_2ch.wav"
    debug_print(f"AUDIO FILE: {file_path}")
    
    # Play the audio
    success = play_stereo_to_separate_outputs(file_path, output_12_id, output_34_id)
    
    if success:
        debug_print("✓ PLAYBACK COMPLETED SUCCESSFULLY")
    else:
        debug_print("⚠️ PLAYBACK ENCOUNTERED ISSUES")

if __name__ == "__main__":
    main()

[02:50:47] STEREO TO DUAL MONO SYNCHRONIZED PLAYBACK
[02:50:47] ========================================
[02:50:47] USING DEVICES:
[02:50:47]   Left channel → Device 21 (Output 1/2)
[02:50:47]   Right channel → Device 20 (Output 3/4)
[02:50:47] AUDIO FILE: C:\Users\cogpsy-vrlab\Desktop\audio sample\participant_00_design_2ch.wav
[02:50:47] ANALYZING AUDIO FILE AND DEVICES:
[02:50:47]   Device sample rate: 44100Hz
[02:50:47]   Loading file: C:\Users\cogpsy-vrlab\Desktop\audio sample\participant_00_design_2ch.wav
[02:50:48]   File loaded: 80412672 samples, 48000Hz, 2 channels, float32
[02:50:48]   Sample rate mismatch: file=48000Hz, device=44100Hz
[02:50:48]   Resampling audio from 48000Hz to 44100Hz...
[02:50:50]   Resampling complete. Original samples: 80412672, New samples: 73879142
[02:50:50]   Data type: float32
[02:50:50]   Creating test signals to verify different channels...
[02:50:50]   Added test tones: 440 Hz (A4) to left, 880 Hz (A5) to right
[02:50:50]   Left channel: 7387914

In [15]:
import sounddevice as sd
import numpy as np
import time
import threading

def debug_print(message, indent=0):
    """Print debug messages with timestamps and optional indentation"""
    timestamp = time.strftime("%H:%M:%S", time.localtime())
    indent_str = "  " * indent
    print(f"[{timestamp}] {indent_str}{message}")

def run_diagnostic_test():
    """Run diagnostic tests to identify audio routing issues"""
    debug_print("AUDIO ROUTING DIAGNOSTIC TEST")
    debug_print("============================")
    
    # Get device info
    output_12_id = 21  # Output 1/2
    output_34_id = 20  # Output 3/4
    
    try:
        device_12_info = sd.query_devices(output_12_id)
        device_34_info = sd.query_devices(output_34_id)
        
        debug_print(f"Device {output_12_id}: {device_12_info['name']}", 1)
        debug_print(f"Device {output_34_id}: {device_34_info['name']}", 1)
        
        # Get sample rate
        sample_rate = int(device_12_info['default_samplerate'])
        debug_print(f"Using sample rate: {sample_rate}Hz", 1)
        
        # Test parameters
        duration = 5.0  # seconds per test
        
        # Create test signals (different frequencies for easy identification)
        t = np.linspace(0, duration, int(duration * sample_rate), endpoint=False)
        
        # First signal: 440 Hz (A4) sine wave
        signal_1 = 0.5 * np.sin(2 * np.pi * 440 * t).astype(np.float32)
        
        # Second signal: 880 Hz (A5) sine wave
        signal_2 = 0.5 * np.sin(2 * np.pi * 880 * t).astype(np.float32)
        
        # Apply fade in/out
        fade_samples = int(0.1 * sample_rate)  # 100ms fade
        fade_in = np.linspace(0, 1, fade_samples)
        fade_out = np.linspace(1, 0, fade_samples)
        
        signal_1[:fade_samples] *= fade_in
        signal_1[-fade_samples:] *= fade_out
        signal_2[:fade_samples] *= fade_in
        signal_2[-fade_samples:] *= fade_out
        
        # Test 1: Only left channel of output 1/2 (completely silent on 3/4)
        debug_print("\nTEST 1: 440 Hz tone ONLY on LEFT channel of Output 1/2")
        debug_print("You should hear sound ONLY from Output 1/2 LEFT speaker", 1)
        debug_print("Output 3/4 should be COMPLETELY SILENT", 1)
        
        # Format test signal for output 1/2 (left channel only)
        test1_out12 = np.column_stack((signal_1, np.zeros_like(signal_1)))
        test1_out34 = np.zeros((len(signal_1), 2), dtype=np.float32)
        
        # Run test 1
        input("Press Enter to start Test 1...")
        
        debug_print("Starting Test 1 in 3 seconds...", 1)
        time.sleep(3)
        debug_print("▶️ Playing 440 Hz tone on LEFT channel of Output 1/2 only", 1)
        
        # Play simultaneously but with silence on device 2
        event = threading.Event()
        
        def play_device1():
            with sd.OutputStream(
                samplerate=sample_rate, 
                blocksize=256,
                device=output_12_id, 
                channels=2, 
                dtype='float32',
                latency='low'
            ) as stream:
                event.wait()
                stream.write(test1_out12)
        
        def play_device2():
            with sd.OutputStream(
                samplerate=sample_rate, 
                blocksize=256,
                device=output_34_id, 
                channels=2, 
                dtype='float32',
                latency='low'
            ) as stream:
                event.wait()
                stream.write(test1_out34)
        
        # Start threads
        thread1 = threading.Thread(target=play_device1)
        thread2 = threading.Thread(target=play_device2)
        
        thread1.start()
        thread2.start()
        
        # Trigger playback
        time.sleep(0.5)
        event.set()
        
        # Wait for playback to complete
        thread1.join()
        thread2.join()
        
        # Ask for feedback
        debug_print("Test 1 complete.", 1)
        result = input("Did you hear sound ONLY from Output 1/2 LEFT and nowhere else? (y/n): ").strip().lower()
        
        if result == 'y':
            debug_print("✓ Test 1 PASSED - Output isolation working correctly", 1)
        else:
            debug_print("⚠️ Test 1 FAILED - Heard audio where there should be silence", 1)
            debug_print("This indicates a routing issue with your audio setup", 1)
        
        # Test 2: Only right channel of output 3/4 (completely silent on 1/2)
        debug_print("\nTEST 2: 880 Hz tone ONLY on RIGHT channel of Output 3/4")
        debug_print("You should hear sound ONLY from Output 3/4 RIGHT speaker", 1)
        debug_print("Output 1/2 should be COMPLETELY SILENT", 1)
        
        # Format test signal for output 3/4 (right channel only)
        test2_out12 = np.zeros((len(signal_2), 2), dtype=np.float32)
        test2_out34 = np.column_stack((np.zeros_like(signal_2), signal_2))
        
        # Run test 2
        input("Press Enter to start Test 2...")
        
        debug_print("Starting Test 2 in 3 seconds...", 1)
        time.sleep(3)
        debug_print("▶️ Playing 880 Hz tone on RIGHT channel of Output 3/4 only", 1)
        
        # Reset event
        event.clear()
        
        # Play simultaneously but with silence on device 1
        def play_device1_test2():
            with sd.OutputStream(
                samplerate=sample_rate, 
                blocksize=256,
                device=output_12_id, 
                channels=2, 
                dtype='float32',
                latency='low'
            ) as stream:
                event.wait()
                stream.write(test2_out12)
        
        def play_device2_test2():
            with sd.OutputStream(
                samplerate=sample_rate, 
                blocksize=256,
                device=output_34_id, 
                channels=2, 
                dtype='float32',
                latency='low'
            ) as stream:
                event.wait()
                stream.write(test2_out34)
        
        # Start threads
        thread1 = threading.Thread(target=play_device1_test2)
        thread2 = threading.Thread(target=play_device2_test2)
        
        thread1.start()
        thread2.start()
        
        # Trigger playback
        time.sleep(0.5)
        event.set()
        
        # Wait for playback to complete
        thread1.join()
        thread2.join()
        
        # Ask for feedback
        debug_print("Test 2 complete.", 1)
        result = input("Did you hear sound ONLY from Output 3/4 RIGHT and nowhere else? (y/n): ").strip().lower()
        
        if result == 'y':
            debug_print("✓ Test 2 PASSED - Output isolation working correctly", 1)
        else:
            debug_print("⚠️ Test 2 FAILED - Heard audio where there should be silence", 1)
            debug_print("This indicates a routing issue with your audio setup", 1)
        
        # Check Komplete Audio 6 MK2 Control Panel
        debug_print("\nTROUBLESHOOTING RECOMMENDATIONS:")
        debug_print("1. Check Komplete Audio 6 MK2 Control Panel for routing settings", 1)
        debug_print("2. Verify Direct Monitoring is fully disabled (knob turned counter-clockwise)", 1)
        debug_print("3. Check Windows Sound Settings for any audio mirroring", 1)
        debug_print("4. Try different USB port or cable", 1)
        debug_print("5. Try disconnecting and reconnecting the interface", 1)
        debug_print("6. Update Komplete Audio 6 MK2 drivers", 1)
        
    except Exception as e:
        debug_print(f"Error during diagnostic test: {e}", 1)

if __name__ == "__main__":
    run_diagnostic_test()

[02:56:49] AUDIO ROUTING DIAGNOSTIC TEST
[02:56:49] ============================
[02:56:49]   Device 21: Output 1/2 (2- Komplete Audio 6 MK2)
[02:56:49]   Device 20: Output 3/4 (2- Komplete Audio 6 MK2)
[02:56:49]   Using sample rate: 44100Hz
[02:56:49] 
TEST 1: 440 Hz tone ONLY on LEFT channel of Output 1/2
[02:56:49]   You should hear sound ONLY from Output 1/2 LEFT speaker
[02:56:49]   Output 3/4 should be COMPLETELY SILENT


Press Enter to start Test 1... 


[02:57:18]   Starting Test 1 in 3 seconds...
[02:57:21]   ▶️ Playing 440 Hz tone on LEFT channel of Output 1/2 only
[02:57:27]   Test 1 complete.


Did you hear sound ONLY from Output 1/2 LEFT and nowhere else? (y/n):  y


[02:57:31]   ✓ Test 1 PASSED - Output isolation working correctly
[02:57:31] 
TEST 2: 880 Hz tone ONLY on RIGHT channel of Output 3/4
[02:57:31]   You should hear sound ONLY from Output 3/4 RIGHT speaker
[02:57:31]   Output 1/2 should be COMPLETELY SILENT


Press Enter to start Test 2... 


[02:57:33]   Starting Test 2 in 3 seconds...
[02:57:36]   ▶️ Playing 880 Hz tone on RIGHT channel of Output 3/4 only
[02:57:42]   Test 2 complete.


KeyboardInterrupt: Interrupted by user

In [ ]:
import sounddevice as sd
import soundfile as sf
import numpy as np
import time
import threading

def debug_print(message, indent=0):
    """Print debug messages with timestamps and optional indentation"""
    timestamp = time.strftime("%H:%M:%S", time.localtime())
    indent_str = "  " * indent
    print(f"[{timestamp}] {indent_str}{message}")

def simple_resample(audio, orig_sr, target_sr):
    """Simple resampling function using linear interpolation"""
    debug_print(f"Resampling audio from {orig_sr}Hz to {target_sr}Hz...", 1)
    
    # Calculate the resampling ratio and new length
    ratio = target_sr / orig_sr
    new_length = int(len(audio) * ratio)
    
    # Create time arrays for interpolation
    orig_time = np.arange(len(audio)) / orig_sr
    new_time = np.arange(new_length) / target_sr
    
    # Create output array - explicitly use float32 for compatibility
    resampled = np.zeros((new_length, audio.shape[1]), dtype=np.float32)
    
    # Resample each channel separately using linear interpolation
    for channel in range(audio.shape[1]):
        resampled[:, channel] = np.interp(new_time, orig_time, audio[:, channel])
    
    debug_print(f"Resampling complete. Original samples: {len(audio)}, New samples: {len(resampled)}", 1)
    return resampled

def list_audio_devices():
    """List all available audio devices grouped by API"""
    debug_print("LISTING ALL AUDIO OUTPUT DEVICES BY API:")
    
    apis = {}
    devices = sd.query_devices()
    
    for i, device in enumerate(devices):
        if device['max_output_channels'] > 0:
            # Determine API type
            api_name = "Unknown"
            if 'hostapi' in device:
                hostapi = device['hostapi']
                if hostapi == 0:
                    api_name = "MME"
                elif hostapi == 1:
                    api_name = "DirectSound"
                elif hostapi == 2:
                    api_name = "WASAPI"
                elif hostapi == 3:
                    api_name = "ASIO"
                else:
                    api_name = f"API {hostapi}"
            
            # Add to dictionary by API
            if api_name not in apis:
                apis[api_name] = []
            
            apis[api_name].append({
                'id': i,
                'name': device['name'],
                'channels': device['max_output_channels'],
                'sr': device.get('default_samplerate', 0),
                'latency': device.get('default_low_output_latency', 0) * 1000
            })
    
    # Print each API group
    for api_name, devices in apis.items():
        debug_print(f"{api_name} DEVICES:", 1)
        for device in devices:
            debug_print(f"ID {device['id']}: {device['name']} - {device['channels']} channels, {device['latency']:.1f}ms", 2)
    
    # Look specifically for ASIO devices
    if "ASIO" in apis:
        debug_print("\nASIO DEVICES FOUND - RECOMMENDED:", 1)
        for device in apis["ASIO"]:
            debug_print(f"ID {device['id']}: {device['name']}", 2)
    
    return apis

def try_asio_playback(file_path, device_id):
    """Try playing the stereo file using ASIO"""
    debug_print(f"ATTEMPTING ASIO PLAYBACK ON DEVICE {device_id}:")
    
    try:
        # Get device info
        device_info = sd.query_devices(device_id)
        debug_print(f"Using device: {device_info['name']}", 1)
        
        device_sr = int(device_info['default_samplerate'])
        debug_print(f"Device sample rate: {device_sr}Hz", 1)
        
        # Load audio file
        debug_print(f"Loading file: {file_path}", 1)
        audio, file_sr = sf.read(file_path, always_2d=True, dtype='float32')
        debug_print(f"File loaded: {len(audio)} samples, {file_sr}Hz, {audio.shape[1]} channels", 1)
        
        # Resample if needed
        if file_sr != device_sr:
            debug_print(f"Sample rate mismatch: file={file_sr}Hz, device={device_sr}Hz", 1)
            audio = simple_resample(audio, file_sr, device_sr)
        
        # Split the channels
        left_channel = audio[:, 0].copy()
        right_channel = audio[:, 1].copy()
        
        # Create test tones for identification
        debug_print("Adding test tones to start of each channel...", 1)
        test_duration = 3  # seconds
        test_samples = int(test_duration * device_sr)
        
        t = np.linspace(0, test_duration, test_samples, endpoint=False)
        test_left = 0.5 * np.sin(2 * np.pi * 440 * t).astype(np.float32)  # 440 Hz (A4)
        test_right = 0.5 * np.sin(2 * np.pi * 880 * t).astype(np.float32)  # 880 Hz (A5)
        
        # Add fade in/out
        fade_samples = int(0.1 * device_sr)
        fade_in = np.linspace(0, 1, fade_samples)
        fade_out = np.linspace(1, 0, fade_samples)
        
        test_left[:fade_samples] *= fade_in
        test_left[-fade_samples:] *= fade_out
        test_right[:fade_samples] *= fade_in
        test_right[-fade_samples:] *= fade_out
        
        # Replace beginning of channels with test tones
        if len(left_channel) > test_samples:
            left_channel[:test_samples] = test_left
            right_channel[:test_samples] = test_right
        
        # Format for 4-channel output if possible
        debug_print(f"Device has {device_info['max_output_channels']} output channels", 1)
        
        if device_info['max_output_channels'] >= 4:
            # For 4+ channel device: route to outputs 1, 2, 3, 4
            debug_print("Using 4-channel mode: Left channel → Output 1, Right channel → Output 3", 1)
            
            # Create a 4-channel output array with zeros
            output_data = np.zeros((len(left_channel), 4), dtype=np.float32)
            
            # Map channels: left → output 1, right → output 3
            output_data[:, 0] = left_channel   # Output 1 (left of 1/2)
            output_data[:, 2] = right_channel  # Output 3 (left of 3/4)
            
        else:
            # For stereo device: use standard stereo output
            debug_print("Using stereo mode: Left → Left, Right → Right", 1)
            output_data = np.column_stack((left_channel, right_channel))
        
        # Start playback
        debug_print("\nINITIATING PLAYBACK:")
        for i in range(3, 0, -1):
            debug_print(f"Starting in {i}...", 1)
            time.sleep(1)
        
        debug_print("⏯️ STARTING PLAYBACK NOW!", 1)
        debug_print("First 3 seconds: 440Hz test tone on Left, 880Hz on Right", 1)
        
        # Use blocking playback for simplicity
        sd.play(output_data, device_sr, device=device_id, blocking=False)
        
        # Show progress
        total_duration = len(output_data) / device_sr
        start_time = time.time()
        
        debug_print(f"Playing {total_duration:.1f} seconds of audio...", 1)
        
        try:
            while sd.get_stream().active:
                elapsed = time.time() - start_time
                percent = min(100, int(elapsed / total_duration * 100))
                debug_print(f"Playback progress: {percent}% ({elapsed:.1f}s / {total_duration:.1f}s)", 1)
                time.sleep(1)
        except KeyboardInterrupt:
            sd.stop()
            debug_print("Playback interrupted by user", 1)
        
        debug_print("Playback complete!", 1)
        return True
        
    except Exception as e:
        debug_print(f"ASIO playback error: {e}", 1)
        return False

def try_shared_mode_playback(file_path, device_left_id, device_right_id):
    """Try playing using shared mode on WASAPI devices"""
    debug_print("ATTEMPTING SHARED MODE WASAPI PLAYBACK:")
    
    try:
        # Get device info
        device_left_info = sd.query_devices(device_left_id)
        device_right_info = sd.query_devices(device_right_id)
        
        debug_print(f"Left device: {device_left_info['name']}", 1)
        debug_print(f"Right device: {device_right_info['name']}", 1)
        
        # Get sample rates
        device_sr = int(device_left_info['default_samplerate'])
        debug_print(f"Using sample rate: {device_sr}Hz", 1)
        
        # Load audio file
        debug_print(f"Loading file: {file_path}", 1)
        audio, file_sr = sf.read(file_path, always_2d=True, dtype='float32')
        debug_print(f"File loaded: {len(audio)} samples, {file_sr}Hz, {audio.shape[1]} channels", 1)
        
        # Resample if needed
        if file_sr != device_sr:
            debug_print(f"Sample rate mismatch: file={file_sr}Hz, device={device_sr}Hz", 1)
            audio = simple_resample(audio, file_sr, device_sr)
        
        # Split the channels
        left_channel = audio[:, 0].copy().astype(np.float32)
        right_channel = audio[:, 1].copy().astype(np.float32)
        
        # Create test tones
        debug_print("Adding test tones to start of each channel...", 1)
        test_duration = 3  # seconds
        test_samples = int(test_duration * device_sr)
        
        t = np.linspace(0, test_duration, test_samples, endpoint=False)
        test_left = 0.5 * np.sin(2 * np.pi * 440 * t).astype(np.float32)  # 440 Hz (A4)
        test_right = 0.5 * np.sin(2 * np.pi * 880 * t).astype(np.float32)  # 880 Hz (A5)
        
        # Add fade in/out
        fade_samples = int(0.1 * device_sr)
        fade_in = np.linspace(0, 1, fade_samples)
        fade_out = np.linspace(1, 0, fade_samples)
        
        test_left[:fade_samples] *= fade_in
        test_left[-fade_samples:] *= fade_out
        test_right[:fade_samples] *= fade_in
        test_right[-fade_samples:] *= fade_out
        
        # Replace beginning of channels with test tones
        if len(left_channel) > test_samples:
            left_channel[:test_samples] = test_left
            right_channel[:test_samples] = test_right
        
        # Format output data
        left_output = np.column_stack((left_channel, np.zeros_like(left_channel)))
        right_output = np.column_stack((np.zeros_like(right_channel), right_channel))
        
        # Try sequential playback
        debug_print("\nATTEMPTING SEQUENTIAL PLAYBACK (wait for first to complete):")
        debug_print("First playing left channel on Output 1/2...", 1)
        
        # Use the first approach - play left channel first
        sd.play(left_output, device_sr, device=device_left_id, blocking=True)
        debug_print("Left channel playback complete", 1)
        
        debug_print("Now playing right channel on Output 3/4...", 1)
        sd.play(right_output, device_sr, device=device_right_id, blocking=True)
        debug_print("Right channel playback complete", 1)
        
        debug_print("\nSequential playback completed successfully!", 1)
        
        return True
        
    except Exception as e:
        debug_print(f"Shared mode playback error: {e}", 1)
        return False

def main():
    """Main function to try different approaches"""
    debug_print("MULTI-APPROACH AUDIO PLAYBACK TEST")
    debug_print("================================")
    
    # File path for your 2-channel WAV
    file_path = r"C:\Users\cogpsy-vrlab\Desktop\audio sample\participant_00_design_2ch.wav"
    debug_print(f"AUDIO FILE: {file_path}")
    
    # List devices to find ASIO and other options
    apis = list_audio_devices()
    
    # Try approach 1: Use ASIO if available
    asio_attempted = False
    if "ASIO" in apis and apis["ASIO"]:
        debug_print("\nAPPROACH 1: TRYING ASIO PLAYBACK")
        # Use first ASIO device (likely your interface)
        asio_device_id = apis["ASIO"][0]['id']
        debug_print(f"Using ASIO device ID {asio_device_id}: {apis['ASIO'][0]['name']}", 1)
        
        asio_success = try_asio_playback(file_path, asio_device_id)
        asio_attempted = True
        
        if asio_success:
            debug_print("✓ ASIO PLAYBACK SUCCESSFUL!")
            return
    
    # If ASIO wasn't available or failed, try approach 2: Sequential playback
    debug_print("\nAPPROACH 2: TRYING SEQUENTIAL PLAYBACK")
    
    # Use the WASAPI devices with lowest latency
    output_12_id = 21  # Output 1/2
    output_34_id = 20  # Output 3/4
    
    shared_success = try_shared_mode_playback(file_path, output_12_id, output_34_id)
    
    if shared_success:
        debug_print("✓ SEQUENTIAL PLAYBACK SUCCESSFUL!")
    else:
        debug_print("⚠️ All playback approaches failed.")
        
        # Provide recommendations
        debug_print("\nRECOMMENDATIONS:")
        if not asio_attempted:
            debug_print("1. Install ASIO drivers for your Komplete Audio 6 MK2", 1)
        debug_print("2. Check Native Instruments Control Center for configuration options", 1)
        debug_print("3. Consider using external audio channel splitting hardware", 1)
        debug_print("4. Try restarting your computer and audio interface", 1)

if __name__ == "__main__":
    main()

[03:02:19] MULTI-APPROACH AUDIO PLAYBACK TEST
[03:02:19] ================================
[03:02:19] AUDIO FILE: C:\Users\cogpsy-vrlab\Desktop\audio sample\participant_00_design_2ch.wav
[03:02:19] LISTING ALL AUDIO OUTPUT DEVICES BY API:
[03:02:19]   MME DEVICES:
[03:02:19]     ID 5: Microsoft Sound Mapper - Output - 2 channels, 90.0ms
[03:02:19]     ID 6: Output 1/2 (2- Komplete Audio 6 - 2 channels, 90.0ms
[03:02:19]     ID 7: Output 3/4 (2- Komplete Audio 6 - 2 channels, 90.0ms
[03:02:19]     ID 8: Speakers (Realtek(R) Audio) - 2 channels, 90.0ms
[03:02:19]     ID 9: SPDIF Output (2- Komplete Audio - 2 channels, 90.0ms
[03:02:19]   DirectSound DEVICES:
[03:02:19]     ID 15: Primary Sound Driver - 2 channels, 120.0ms
[03:02:19]     ID 16: Output 1/2 (2- Komplete Audio 6 MK2) - 2 channels, 120.0ms
[03:02:19]     ID 17: Output 3/4 (2- Komplete Audio 6 MK2) - 2 channels, 120.0ms
[03:02:19]     ID 18: Speakers (Realtek(R) Audio) - 2 channels, 120.0ms
[03:02:19]     ID 19: SPDIF Output (2

In [ ]:
import sounddevice as sd
import soundfile as sf
import numpy as np
import time

def debug_print(message, indent=0):
    """Print debug messages with timestamps and optional indentation"""
    timestamp = time.strftime("%H:%M:%S", time.localtime())
    indent_str = "  " * indent
    print(f"[{timestamp}] {indent_str}{message}")

def simple_resample(audio, orig_sr, target_sr):
    """Simple resampling function using linear interpolation"""
    debug_print(f"Resampling audio from {orig_sr}Hz to {target_sr}Hz...", 1)
    
    # Calculate the resampling ratio and new length
    ratio = target_sr / orig_sr
    new_length = int(len(audio) * ratio)
    
    # Create time arrays for interpolation
    orig_time = np.arange(len(audio)) / orig_sr
    new_time = np.arange(new_length) / target_sr
    
    # Create output array - explicitly use float32 for compatibility
    resampled = np.zeros((new_length, audio.shape[1]), dtype=np.float32)
    
    # Resample each channel separately using linear interpolation
    for channel in range(audio.shape[1]):
        resampled[:, channel] = np.interp(new_time, orig_time, audio[:, channel])
    
    debug_print(f"Resampling complete. Original samples: {len(audio)}, New samples: {len(resampled)}", 1)
    return resampled

def try_multi_channel_device(file_path):
    """Try to find a multi-channel ASIO device and use it for playback"""
    debug_print("SEARCHING FOR MULTI-CHANNEL ASIO DEVICES:")
    
    # First look for Native Instruments ASIO driver (should be multi-channel)
    ni_asio_id = None
    ni_asio_name = None
    best_channel_count = 0
    
    # Also look for any device with 4 or more channels
    multi_channel_id = None
    multi_channel_name = None
    
    devices = sd.query_devices()
    
    for i, device in enumerate(devices):
        if device['max_output_channels'] > 0:
            # Check if it's an ASIO device
            api_name = "Unknown"
            if 'hostapi' in device:
                hostapi = device['hostapi']
                if hostapi == 3:  # ASIO
                    debug_print(f"Found ASIO device {i}: {device['name']} with {device['max_output_channels']} channels", 1)
                    
                    # Check if it's a Native Instruments device
                    if "komplete" in device['name'].lower() or "native instruments" in device['name'].lower():
                        debug_print(f"Found Native Instruments ASIO device: {device['name']}", 1)
                        ni_asio_id = i
                        ni_asio_name = device['name']
                        break
                    
                    # Otherwise, track the device with most channels
                    if device['max_output_channels'] >= 4 and device['max_output_channels'] > best_channel_count:
                        best_channel_count = device['max_output_channels']
                        multi_channel_id = i
                        multi_channel_name = device['name']
    
    # Use Native Instruments ASIO if found, otherwise try the best multi-channel device
    if ni_asio_id is not None:
        debug_print(f"Using Native Instruments ASIO device: {ni_asio_name}", 1)
        asio_device_id = ni_asio_id
    elif multi_channel_id is not None:
        debug_print(f"Using multi-channel ASIO device: {multi_channel_name} ({best_channel_count} channels)", 1)
        asio_device_id = multi_channel_id
    else:
        debug_print("No suitable multi-channel ASIO device found", 1)
        return False
    
    # Try playing through the selected device
    try:
        debug_print(f"Loading file: {file_path}", 1)
        audio, file_sr = sf.read(file_path, always_2d=True, dtype='float32')
        debug_print(f"File loaded: {len(audio)} samples, {file_sr}Hz, {audio.shape[1]} channels", 1)
        
        # Get device sample rate
        device_info = sd.query_devices(asio_device_id)
        device_sr = int(device_info['default_samplerate'])
        debug_print(f"Device sample rate: {device_sr}Hz", 1)
        
        # Resample if needed
        if file_sr != device_sr:
            debug_print(f"Sample rate mismatch: file={file_sr}Hz, device={device_sr}Hz", 1)
            audio = simple_resample(audio, file_sr, device_sr)
        
        # Create multi-channel output array based on device capabilities
        channel_count = min(device_info['max_output_channels'], 8)  # Cap at 8 channels for safety
        output_array = np.zeros((len(audio), channel_count), dtype=np.float32)
        
        # Map stereo channels to first and third outputs
        debug_print(f"Routing left channel to output 1, right channel to output 3", 1)
        output_array[:, 0] = audio[:, 0]  # Left to output 1
        
        if channel_count >= 3:
            output_array[:, 2] = audio[:, 1]  # Right to output 3
        else:
            output_array[:, 1] = audio[:, 1]  # Right to output 2 if only 2 channels
        
        # Start playback
        debug_print("\nINITIATING ASIO MULTI-CHANNEL PLAYBACK:")
        for i in range(3, 0, -1):
            debug_print(f"Starting in {i}...", 1)
            time.sleep(1)
        
        debug_print("⏯️ STARTING PLAYBACK NOW!", 1)
        
        sd.play(output_array, device_sr, device=asio_device_id, blocking=False)
        
        # Show progress
        total_duration = len(audio) / device_sr
        start_time = time.time()
        
        debug_print(f"Playing {total_duration:.1f} seconds of audio...", 1)
        
        try:
            while sd.get_stream().active:
                elapsed = time.time() - start_time
                percent = min(100, int(elapsed / total_duration * 100))
                debug_print(f"Playback progress: {percent}% ({elapsed:.1f}s / {total_duration:.1f}s)", 1)
                time.sleep(5)  # Update every 5 seconds for long files
        except KeyboardInterrupt:
            sd.stop()
            debug_print("Playback interrupted by user", 1)
        
        debug_print("Playback complete!", 1)
        return True
        
    except Exception as e:
        debug_print(f"Error during multi-channel playback: {e}", 1)
        return False

def try_using_specific_komplete_asio(file_path):
    """Try using the specific Komplete Audio ASIO driver"""
    debug_print("TRYING SPECIFIC KOMPLETE AUDIO ASIO DEVICES:")
    
    # Get device list
    devices = sd.query_devices()
    
    # Look for Komplete Audio ASIO devices
    output_12_id = None
    output_34_id = None
    
    for i, device in enumerate(devices):
        if device['max_output_channels'] > 0:
            # Check if it's an ASIO device
            if 'hostapi' in device and device['hostapi'] == 3:  # ASIO
                name = device['name'].lower()
                if "output 1/2" in name and ("komplete" in name or "1/2" in name):
                    output_12_id = i
                    debug_print(f"Found Output 1/2 ASIO device: ID {i} - {device['name']}", 1)
                elif "output 3/4" in name and ("komplete" in name or "3/4" in name):
                    output_34_id = i
                    debug_print(f"Found Output 3/4 ASIO device: ID {i} - {device['name']}", 1)
    
    # Check if we found both devices
    if output_12_id is None or output_34_id is None:
        debug_print("Could not find both ASIO devices for Komplete Audio", 1)
        return False
    
    # Try the specific IDs we found from the log
    if output_12_id is None:
        output_12_id = 42  # From the log
    if output_34_id is None:
        output_34_id = 41  # From the log
    
    debug_print(f"Using Output 1/2: Device {output_12_id}", 1)
    debug_print(f"Using Output 3/4: Device {output_34_id}", 1)
    
    # Try to play through both ASIO devices
    try:
        # Load file
        debug_print(f"Loading file: {file_path}", 1)
        audio, file_sr = sf.read(file_path, always_2d=True, dtype='float32')
        debug_print(f"File loaded: {len(audio)} samples, {file_sr}Hz, {audio.shape[1]} channels", 1)
        
        # Get device sample rate (assume same for both)
        try:
            device_info = sd.query_devices(output_12_id)
            device_sr = int(device_info['default_samplerate'])
        except:
            device_sr = 44100  # Default to 44.1kHz if query fails
        
        debug_print(f"Using sample rate: {device_sr}Hz", 1)
        
        # Resample if needed
        if file_sr != device_sr:
            debug_print(f"Sample rate mismatch: file={file_sr}Hz, device={device_sr}Hz", 1)
            audio = simple_resample(audio, file_sr, device_sr)
        
        # Split channels and create mono output for each device
        left_channel = audio[:, 0].copy()
        right_channel = audio[:, 1].copy()
        
        # Add test signals to beginning to verify routing
        test_duration = 5  # seconds
        test_samples = int(test_duration * device_sr)
        
        t = np.linspace(0, test_duration, test_samples, endpoint=False)
        test_left = 0.5 * np.sin(2 * np.pi * 440 * t).astype(np.float32)  # 440 Hz (A4)
        test_right = 0.5 * np.sin(2 * np.pi * 880 * t).astype(np.float32)  # 880 Hz (A5)
        
        # Add fade in/out
        fade_samples = int(0.1 * device_sr)
        fade_in = np.linspace(0, 1, fade_samples)
        fade_out = np.linspace(1, 0, fade_samples)
        
        test_left[:fade_samples] *= fade_in
        test_left[-fade_samples:] *= fade_out
        test_right[:fade_samples] *= fade_in
        test_right[-fade_samples:] *= fade_out
        
        # Replace beginning of channels with test tones
        if len(left_channel) > test_samples:
            left_channel[:test_samples] = test_left
            right_channel[:test_samples] = test_right
        
        # Create stereo output for each device (we need to send stereo)
        left_output = np.column_stack((left_channel, left_channel)).astype(np.float32)
        right_output = np.column_stack((right_channel, right_channel)).astype(np.float32)
        
        # Set up ASIO playback - we need to open both devices
        debug_print("\nINITIATING DIRECT ASIO PLAYBACK:")
        for i in range(3, 0, -1):
            debug_print(f"Starting in {i}...", 1)
            time.sleep(1)
        
        debug_print("⏯️ STARTING PLAYBACK NOW!", 1)
        debug_print("First 5 seconds: Test tones (440Hz on Output 1/2, 880Hz on Output 3/4)", 1)
        
        # On some systems we can do this with separate threads
        import threading
        
        def play_left():
            try:
                sd.play(left_output, device_sr, device=output_12_id, blocking=True)
                debug_print("Output 1/2 playback complete", 1)
            except Exception as e:
                debug_print(f"Error on Output 1/2: {e}", 1)
        
        def play_right():
            try:
                sd.play(right_output, device_sr, device=output_34_id, blocking=True)
                debug_print("Output 3/4 playback complete", 1)
            except Exception as e:
                debug_print(f"Error on Output 3/4: {e}", 1)
        
        thread_left = threading.Thread(target=play_left)
        thread_right = threading.Thread(target=play_right)
        
        thread_left.start()
        thread_right.start()
        
        # Wait for one thread to complete
        total_duration = len(audio) / device_sr
        start_time = time.time()
        
        debug_print(f"Playing {total_duration:.1f} seconds of audio...", 1)
        
        # Show progress
        try:
            while thread_left.is_alive() or thread_right.is_alive():
                elapsed = time.time() - start_time
                percent = min(100, int(elapsed / total_duration * 100))
                debug_print(f"Playback progress: {percent}% ({elapsed:.1f}s / {total_duration:.1f}s)", 1)
                time.sleep(5)  # Update every 5 seconds for long files
        except KeyboardInterrupt:
            sd.stop()
            debug_print("Playback interrupted by user", 1)
        
        debug_print("Playback complete!", 1)
        return True
        
    except Exception as e:
        debug_print(f"Error in ASIO direct playback: {e}", 1)
        return False

def try_multi_channel_routing(file_path):
    """Try using a 8-channel device to route properly"""
    debug_print("TRYING MULTI-CHANNEL ROUTING:")
    
    # Look for multi-channel device (specifically Nahimic Easy Surround with 8 channels)
    multi_channel_id = None
    
    devices = sd.query_devices()
    
    for i, device in enumerate(devices):
        if device['max_output_channels'] >= 4:
            debug_print(f"Found multi-channel device: ID {i} - {device['name']} with {device['max_output_channels']} channels", 1)
            if device['max_output_channels'] >= 8 and "surround" in device['name'].lower():
                multi_channel_id = i
                debug_print(f"Selected surround device: {device['name']}", 1)
                break
    
    if multi_channel_id is None:
        for i, device in enumerate(devices):
            if device['max_output_channels'] >= 4:
                multi_channel_id = i
                debug_print(f"Selected first multi-channel device: {device['name']}", 1)
                break
    
    if multi_channel_id is None:
        debug_print("No multi-channel device found", 1)
        return False
    
    try:
        # Load file
        debug_print(f"Loading file: {file_path}", 1)
        audio, file_sr = sf.read(file_path, always_2d=True, dtype='float32')
        debug_print(f"File loaded: {len(audio)} samples, {file_sr}Hz, {audio.shape[1]} channels", 1)
        
        # Get device info
        device_info = sd.query_devices(multi_channel_id)
        device_sr = int(device_info['default_samplerate'])
        channel_count = device_info['max_output_channels']
        
        debug_print(f"Device: {device_info['name']}", 1)
        debug_print(f"Sample rate: {device_sr}Hz", 1)
        debug_print(f"Channels: {channel_count}", 1)
        
        # Resample if needed
        if file_sr != device_sr:
            debug_print(f"Sample rate mismatch: file={file_sr}Hz, device={device_sr}Hz", 1)
            audio = simple_resample(audio, file_sr, device_sr)
        
        # Create multi-channel output array
        output_array = np.zeros((len(audio), channel_count), dtype=np.float32)
        
        # Map stereo channels to specific outputs
        # For 5.1/7.1 surround:
        # 0 = Front Left, 1 = Front Right
        # 2 = Center, 3 = LFE
        # 4 = Rear Left, 5 = Rear Right
        # 6 = Side Left, 7 = Side Right
        
        debug_print("Mapping left channel to outputs 0, 4, 6", 1)
        debug_print("Mapping right channel to outputs 1, 5, 7", 1)
        
        # Map left to front left, rear left, side left
        output_array[:, 0] = audio[:, 0]  # Front Left
        if channel_count > 4:
            output_array[:, 4] = audio[:, 0]  # Rear Left
        if channel_count > 6:
            output_array[:, 6] = audio[:, 0]  # Side Left
            
        # Map right to front right, rear right, side right
        output_array[:, 1] = audio[:, 1]  # Front Right
        if channel_count > 5:
            output_array[:, 5] = audio[:, 1]  # Rear Right
        if channel_count > 7:
            output_array[:, 7] = audio[:, 1]  # Side Right
        
        # Start playback
        debug_print("\nINITIATING MULTI-CHANNEL SURROUND PLAYBACK:")
        for i in range(3, 0, -1):
            debug_print(f"Starting in {i}...", 1)
            time.sleep(1)
        
        debug_print("⏯️ STARTING PLAYBACK NOW!", 1)
        
        sd.play(output_array, device_sr, device=multi_channel_id, blocking=False)
        
        # Show progress
        total_duration = len(audio) / device_sr
        start_time = time.time()
        
        debug_print(f"Playing {total_duration:.1f} seconds of audio...", 1)
        
        try:
            while sd.get_stream().active:
                elapsed = time.time() - start_time
                percent = min(100, int(elapsed / total_duration * 100))
                debug_print(f"Playback progress: {percent}% ({elapsed:.1f}s / {total_duration:.1f}s)", 1)
                time.sleep(5)  # Update every 5 seconds for long files
        except KeyboardInterrupt:
            sd.stop()
            debug_print("Playback interrupted by user", 1)
        
        debug_print("Playback complete!", 1)
        return True
        
    except Exception as e:
        debug_print(f"Error in multi-channel routing: {e}", 1)
        return False

def main():
    """Main function to try different approaches"""
    debug_print("KOMPLETE AUDIO 6 ASIO DIRECT PLAYBACK")
    debug_print("====================================")
    
    # File path for your 2-channel WAV
    file_path = r"C:\Users\cogpsy-vrlab\Desktop\audio sample\participant_00_design_2ch.wav"
    debug_print(f"AUDIO FILE: {file_path}")
    
    # Try approach 1: Use ASIO devices directly
    debug_print("\nAPPROACH 1: USING KOMPLETE AUDIO ASIO DEVICES")
    if try_using_specific_komplete_asio(file_path):
        debug_print("✓ ASIO DIRECT PLAYBACK SUCCESSFUL!")
        return
    
    # If that fails, try approach 2: Use a multi-channel device
    debug_print("\nAPPROACH 2: TRYING MULTI-CHANNEL DEVICE")
    if try_multi_channel_device(file_path):
        debug_print("✓ MULTI-CHANNEL PLAYBACK SUCCESSFUL!")
        return
    
    # If that fails too, try approach 3: Try surround sound routing
    debug_print("\nAPPROACH 3: TRYING SURROUND SOUND ROUTING")
    if try_multi_channel_routing(file_path):
        debug_print("✓ SURROUND SOUND ROUTING SUCCESSFUL!")
        return
    
    # If all approaches fail
    debug_print("\n⚠️ ALL PLAYBACK APPROACHES FAILED")
    debug_print("\nRECOMMENDATIONS:")
    debug_print("1. Install the Native Instruments ASIO driver from their website", 1)
    debug_print("2. Consider using an external hardware solution like a mixer", 1)
    debug_print("3. Try using a different audio interface that supports multiple outputs", 1)

if __name__ == "__main__":
    main()

[10:55:44] KOMPLETE AUDIO 6 ASIO DIRECT PLAYBACK
[10:55:44] ====================================
[10:55:44] AUDIO FILE: C:\Users\cogpsy-vrlab\Desktop\audio sample\participant_00_design_2ch.wav
[10:55:44] 
APPROACH 1: USING KOMPLETE AUDIO ASIO DEVICES
[10:55:44] TRYING SPECIFIC KOMPLETE AUDIO ASIO DEVICES:
[10:55:44]   Found Output 3/4 ASIO device: ID 41 - Output 3/4 (Output 3/4)
[10:55:44]   Found Output 1/2 ASIO device: ID 42 - Output 1/2 (Output 1/2)
[10:55:44]   Using Output 1/2: Device 42
[10:55:44]   Using Output 3/4: Device 41
[10:55:44]   Loading file: C:\Users\cogpsy-vrlab\Desktop\audio sample\participant_00_design_2ch.wav
[10:55:45]   File loaded: 80412672 samples, 48000Hz, 2 channels
[10:55:45]   Using sample rate: 44100Hz
[10:55:45]   Sample rate mismatch: file=48000Hz, device=44100Hz
[10:55:45]   Resampling audio from 48000Hz to 44100Hz...
[10:55:48]   Resampling complete. Original samples: 80412672, New samples: 73879142
[10:55:49] 
INITIATING DIRECT ASIO PLAYBACK:
[10:55:

In [ ]:
import sounddevice as sd
import soundfile as sf
import numpy as np
import threading
import time

def debug_print(message, indent=0):
    """Print debug messages with timestamps and optional indentation"""
    timestamp = time.strftime("%H:%M:%S", time.localtime())
    indent_str = "  " * indent
    print(f"[{timestamp}] {indent_str}{message}")

def simple_resample(audio, orig_sr, target_sr):
    """Simple resampling function using linear interpolation"""
    debug_print(f"Resampling audio from {orig_sr}Hz to {target_sr}Hz...", 1)
    
    # Calculate the resampling ratio and new length
    ratio = target_sr / orig_sr
    new_length = int(len(audio) * ratio)
    
    # Create time arrays for interpolation
    orig_time = np.arange(len(audio)) / orig_sr
    new_time = np.arange(new_length) / target_sr
    
    # Create output array - explicitly use float32 for compatibility
    resampled = np.zeros((new_length, audio.shape[1]), dtype=np.float32)
    
    # Resample each channel separately using linear interpolation
    for channel in range(audio.shape[1]):
        resampled[:, channel] = np.interp(new_time, orig_time, audio[:, channel])
    
    debug_print(f"Resampling complete. Original samples: {len(audio)}, New samples: {len(resampled)}", 1)
    return resampled

def max_sync_playback(file_path, use_asio=True):
    """Maximum synchronization playback for Komplete Audio 6"""
    debug_print("PREPARING MAXIMUM SYNC PLAYBACK:")
    
    # First identify the devices to use - try ASIO first, then WASAPI
    output_12_id = None
    output_34_id = None
    devices = sd.query_devices()
    
    # Look for devices based on their name and type
    for i, device in enumerate(devices):
        if device['max_output_channels'] > 0:
            # Determine device type
            api_type = "Unknown"
            if 'hostapi' in device:
                hostapi = device['hostapi']
                if hostapi == 0:
                    api_type = "MME"
                elif hostapi == 1:
                    api_type = "DirectSound"
                elif hostapi == 2:
                    api_type = "WASAPI"
                elif hostapi == 3:
                    api_type = "ASIO"
            
            name = device['name'].lower()
            
            # Check if it's a Komplete device with the right API
            is_komplete = "komplete" in name
            is_target_api = (use_asio and api_type == "ASIO") or (not use_asio and api_type == "WASAPI")
            
            if is_komplete and is_target_api:
                if "1/2" in name or "output 1" in name.lower():
                    output_12_id = i
                    debug_print(f"Found Output 1/2 device: ID {i} - {device['name']} ({api_type})", 1)
                elif "3/4" in name or "output 3" in name.lower():
                    output_34_id = i
                    debug_print(f"Found Output 3/4 device: ID {i} - {device['name']} ({api_type})", 1)
    
    # Use hardcoded IDs from previous logs if not found
    if output_12_id is None:
        if use_asio:
            output_12_id = 42  # ASIO
        else:
            output_12_id = 21  # WASAPI
        debug_print(f"Using hardcoded ID {output_12_id} for Output 1/2", 1)
    
    if output_34_id is None:
        if use_asio:
            output_34_id = 41  # ASIO
        else:
            output_34_id = 20  # WASAPI
        debug_print(f"Using hardcoded ID {output_34_id} for Output 3/4", 1)
    
    # Get devices info
    dev_12_info = sd.query_devices(output_12_id)
    dev_34_info = sd.query_devices(output_34_id)
    
    debug_print(f"Using devices:", 1)
    debug_print(f"1/2: {dev_12_info['name']}", 1)
    debug_print(f"3/4: {dev_34_info['name']}", 1)
    
    # Check if there's different latency between devices
    dev_12_latency = dev_12_info.get('default_low_output_latency', 0) * 1000
    dev_34_latency = dev_34_info.get('default_low_output_latency', 0) * 1000
    latency_diff = abs(dev_12_latency - dev_34_latency)
    
    debug_print(f"Device 1/2 latency: {dev_12_latency:.2f}ms", 1)
    debug_print(f"Device 3/4 latency: {dev_34_latency:.2f}ms", 1)
    
    if latency_diff > 0.1:  # More than 0.1ms difference
        debug_print(f"⚠️ Latency difference detected: {latency_diff:.2f}ms", 1)
        debug_print(f"Will apply compensation for precise sync", 1)
    
    try:
        # Load the audio file
        debug_print(f"Loading file: {file_path}", 1)
        audio, file_sr = sf.read(file_path, always_2d=True, dtype='float32')
        debug_print(f"File loaded: {len(audio)} samples, {file_sr}Hz, {audio.shape[1]} channels", 1)
        
        # Get device sample rate
        device_sr = int(dev_12_info['default_samplerate'])
        debug_print(f"Device sample rate: {device_sr}Hz", 1)
        
        # Resample if needed
        if file_sr != device_sr:
            debug_print(f"Sample rate mismatch: file={file_sr}Hz, device={device_sr}Hz", 1)
            audio = simple_resample(audio, file_sr, device_sr)
        
        # Split channels
        left_channel = audio[:, 0].copy()
        right_channel = audio[:, 1].copy()
        
        # Add test signals to beginning to verify routing and sync
        test_duration = 5  # seconds
        test_samples = int(test_duration * device_sr)
        
        # Create precisely synchronized test tones with clear timing markers
        t = np.linspace(0, test_duration, test_samples, endpoint=False)
        
        # Left channel: 440 Hz sine with brief pulses every second
        test_left = 0.7 * np.sin(2 * np.pi * 440 * t).astype(np.float32)
        # Add timing pulses (brief silence) every second
        for i in range(5):
            pulse_pos = int(i * device_sr)
            if pulse_pos + 100 < len(test_left):
                test_left[pulse_pos:pulse_pos+100] = 0
        
        # Right channel: 880 Hz sine with brief pulses every second
        test_right = 0.7 * np.sin(2 * np.pi * 880 * t).astype(np.float32)
        # Add timing pulses (brief silence) every second
        for i in range(5):
            pulse_pos = int(i * device_sr)
            if pulse_pos + 100 < len(test_right):
                test_right[pulse_pos:pulse_pos+100] = 0
        
        # Add fade in/out
        fade_samples = int(0.1 * device_sr)
        fade_in = np.linspace(0, 1, fade_samples)
        fade_out = np.linspace(1, 0, fade_samples)
        
        test_left[:fade_samples] *= fade_in
        test_left[-fade_samples:] *= fade_out
        test_right[:fade_samples] *= fade_in
        test_right[-fade_samples:] *= fade_out
        
        # Replace beginning of channels with test tones
        if len(left_channel) > test_samples:
            left_channel[:test_samples] = test_left
            right_channel[:test_samples] = test_right
        
        # Create stereo outputs with proper routing - EXACTLY as requested
        # Left channel to BOTH left and right speakers of device 1/2
        left_output = np.column_stack((left_channel, left_channel)).astype(np.float32)
        
        # Right channel to BOTH left and right speakers of device 3/4
        right_output = np.column_stack((right_channel, right_channel)).astype(np.float32)
        
        debug_print("CHANNEL ROUTING:", 1) 
        debug_print("Left channel → Output 1/2 (both speakers)", 1)
        debug_print("Right channel → Output 3/4 (both speakers)", 1)
        
        # Ensure float32 format
        left_output = left_output.astype(np.float32)
        right_output = right_output.astype(np.float32)
        
        debug_print("READY FOR SYNCHRONIZATION:", 1)
        debug_print("First 5 seconds: Test tones with timing pulses every second", 1)
        debug_print("Listen carefully to verify precise synchronization", 1)
        
        # Set optimal buffer size for best sync - use small buffers
        blocksize = 128  # Very small buffer size for tight synchronization
        
        # Use a barrier for precise thread synchronization
        sync_barrier = threading.Barrier(3)  # 2 playback threads + main thread
        
        # Threads for synchronized playback
        def play_left():
            try:
                # Create output stream but don't start yet
                with sd.OutputStream(
                    samplerate=device_sr,
                    blocksize=blocksize,
                    device=output_12_id,
                    channels=2,
                    dtype='float32',
                    latency='low'  # Use low latency setting
                ) as stream_left:
                    
                    # Wait at the barrier for sync
                    debug_print("Left output ready and waiting...", 2)
                    sync_barrier.wait()
                    
                    # Exact timestamp for debugging
                    start_time = time.time()
                    debug_print(f"Left output starting at {start_time:.9f}", 2)
                    
                    # Write in small chunks for better monitoring
                    chunk_size = blocksize * 8  # Process multiple blocks at a time
                    total_chunks = len(left_output) // chunk_size + 1
                    
                    for i in range(total_chunks):
                        start_idx = i * chunk_size
                        end_idx = min(start_idx + chunk_size, len(left_output))
                        
                        if start_idx >= len(left_output):
                            break
                        
                        # Get the next chunk
                        chunk = left_output[start_idx:end_idx]
                        
                        # Write to the stream
                        stream_left.write(chunk)
                        
                        # Occasional progress update (not too frequent)
                        if i % 1000 == 0:
                            progress = start_idx / len(left_output) * 100
                            debug_print(f"Left output progress: {progress:.1f}%", 3)
                    
                    debug_print("Left output playback complete", 2)
            except Exception as e:
                debug_print(f"Error in left output: {e}", 2)
        
        def play_right():
            try:
                # Create output stream but don't start yet
                with sd.OutputStream(
                    samplerate=device_sr,
                    blocksize=blocksize,
                    device=output_34_id,
                    channels=2,
                    dtype='float32',
                    latency='low'  # Use low latency setting
                ) as stream_right:
                    
                    # Wait at the barrier for sync
                    debug_print("Right output ready and waiting...", 2)
                    sync_barrier.wait()
                    
                    # Exact timestamp for debugging
                    start_time = time.time()
                    debug_print(f"Right output starting at {start_time:.9f}", 2)
                    
                    # Write in small chunks for better monitoring
                    chunk_size = blocksize * 8  # Process multiple blocks at a time
                    total_chunks = len(right_output) // chunk_size + 1
                    
                    for i in range(total_chunks):
                        start_idx = i * chunk_size
                        end_idx = min(start_idx + chunk_size, len(right_output))
                        
                        if start_idx >= len(right_output):
                            break
                        
                        # Get the next chunk
                        chunk = right_output[start_idx:end_idx]
                        
                        # Write to the stream
                        stream_right.write(chunk)
                        
                        # Occasional progress update (not too frequent)
                        if i % 1000 == 0:
                            progress = start_idx / len(right_output) * 100
                            debug_print(f"Right output progress: {progress:.1f}%", 3)
                    
                    debug_print("Right output playback complete", 2)
            except Exception as e:
                debug_print(f"Error in right output: {e}", 2)
        
        # Start both threads
        debug_print("\nINITIATING MAXIMUM SYNC PLAYBACK:")
        
        # Create threads
        thread_left = threading.Thread(target=play_left)
        thread_right = threading.Thread(target=play_right)
        
        debug_print("Starting playback threads...", 1)
        thread_left.start()
        thread_right.start()
        
        # Precise countdown for synchronized start
        debug_print("Preparing synchronized start...", 1)
        time.sleep(1)  # Give threads time to initialize
        
        # Final countdown with precise timing
        for i in range(3, 0, -1):
            debug_print(f"Starting in {i}...", 1)
            time.sleep(1)
        
        debug_print("⏯️ TRIGGERING PRECISELY SYNCHRONIZED PLAYBACK!", 1)
        
        # This will release both threads EXACTLY at the same time
        start_time = time.time()
        sync_barrier.wait()
        
        debug_print(f"Synchronized trigger at {start_time:.9f}", 1)
        
        # Monitor playback
        total_duration = len(audio) / device_sr
        debug_print(f"Playing {total_duration:.1f} seconds of audio...", 1)
        
        # Wait for threads to complete with progress display
        start_time = time.time()
        try:
            while thread_left.is_alive() or thread_right.is_alive():
                elapsed = time.time() - start_time
                percent = min(100, int(elapsed / total_duration * 100))
                
                # Update every 5 seconds for long files
                time.sleep(5)
                debug_print(f"Playback progress: {percent}% ({elapsed:.1f}s / {total_duration:.1f}s)", 1)
                
        except KeyboardInterrupt:
            debug_print("Playback interrupted by user", 1)
            sd.stop()
        
        # Wait for both threads to finish
        thread_left.join()
        thread_right.join()
        
        debug_print("✓ SYNCHRONIZED PLAYBACK COMPLETE", 1)
        return True
        
    except Exception as e:
        debug_print(f"Error during synchronized playback: {e}", 1)
        return False

def main():
    """Main function to try both ASIO and WASAPI approaches"""
    debug_print("DUAL MONO OUTPUT ROUTING FOR KOMPLETE AUDIO 6")
    debug_print("============================================")
    
    # File path for your 2-channel WAV
    file_path = r"C:\Users\cogpsy-vrlab\Desktop\audio sample\participant_00_design_2ch.wav"
    debug_print(f"AUDIO FILE: {file_path}")
    
    # Try ASIO first (usually better performance)
    debug_print("\nAPPROACH 1: TRYING ASIO DEVICES")
    asio_success = max_sync_playback(file_path, use_asio=True)
    
    # If ASIO failed, try WASAPI
    if not asio_success:
        debug_print("\nAPPROACH 2: TRYING WASAPI DEVICES")
        wasapi_success = max_sync_playback(file_path, use_asio=False)
        
        if not wasapi_success:
            debug_print("\n⚠️ ALL PLAYBACK APPROACHES FAILED")
            debug_print("\nRECOMMENDATIONS:")
            debug_print("1. Check Native Instruments Control Center for device settings", 1)
            debug_print("2. Try updating audio drivers and firmware", 1)
            debug_print("3. Consider external hardware for audio routing", 1)

if __name__ == "__main__":
    main()

[03:18:25] DUAL MONO OUTPUT ROUTING FOR KOMPLETE AUDIO 6
[03:18:25] ============================================
[03:18:25] AUDIO FILE: C:\Users\cogpsy-vrlab\Desktop\audio sample\participant_00_design_2ch.wav
[03:18:25] 
APPROACH 1: TRYING ASIO DEVICES
[03:18:25] PREPARING MAXIMUM SYNC PLAYBACK:
[03:18:25]   Using hardcoded ID 42 for Output 1/2
[03:18:25]   Using hardcoded ID 41 for Output 3/4
[03:18:25]   Using devices:
[03:18:25]   1/2: Output 1/2 (Output 1/2)
[03:18:25]   3/4: Output 3/4 (Output 3/4)
[03:18:25]   Device 1/2 latency: 10.00ms
[03:18:25]   Device 3/4 latency: 10.00ms
[03:18:25]   Loading file: C:\Users\cogpsy-vrlab\Desktop\audio sample\participant_00_design_2ch.wav
[03:18:25]   File loaded: 80412672 samples, 48000Hz, 2 channels
[03:18:25]   Device sample rate: 44100Hz
[03:18:25]   Sample rate mismatch: file=48000Hz, device=44100Hz
[03:18:25]   Resampling audio from 48000Hz to 44100Hz...
[03:18:27]   Resampling complete. Original samples: 80412672, New samples: 73879142

In [1]:
import sounddevice as sd
import soundfile as sf
import numpy as np
import threading
import time

def debug_print(message, indent=0):
    """Print debug messages with timestamps and optional indentation"""
    timestamp = time.strftime("%H:%M:%S", time.localtime())
    indent_str = "  " * indent
    print(f"[{timestamp}] {indent_str}{message}")

def simple_resample(audio, orig_sr, target_sr):
    """Simple resampling function using linear interpolation"""
    debug_print(f"Resampling audio from {orig_sr}Hz to {target_sr}Hz...", 1)
    
    # Calculate the resampling ratio and new length
    ratio = target_sr / orig_sr
    new_length = int(len(audio) * ratio)
    
    # Create time arrays for interpolation
    orig_time = np.arange(len(audio)) / orig_sr
    new_time = np.arange(new_length) / target_sr
    
    # Create output array - explicitly use float32 for compatibility
    resampled = np.zeros((new_length, audio.shape[1]), dtype=np.float32)
    
    # Resample each channel separately using linear interpolation
    for channel in range(audio.shape[1]):
        resampled[:, channel] = np.interp(new_time, orig_time, audio[:, channel])
    
    debug_print(f"Resampling complete. Original samples: {len(audio)}, New samples: {len(resampled)}", 1)
    return resampled

def synchronized_playback():
    """Simplified synchronized playback using WASAPI devices"""
    debug_print("SIMPLE SYNCHRONIZED PLAYBACK")
    
    # Use the WASAPI devices that we know work
    output_12_id = 21  # Output 1/2 WASAPI
    output_34_id = 20  # Output 3/4 WASAPI
    
    debug_print(f"Using devices:", 1)
    debug_print(f"Output 1/2: Device ID {output_12_id}", 1)
    debug_print(f"Output 3/4: Device ID {output_34_id}", 1)
    
    # File path for your 2-channel WAV
    file_path = r"C:\Users\cogpsy-vrlab\Desktop\audio sample\participant_00_design_2ch.wav"
    debug_print(f"Loading file: {file_path}", 1)
    
    try:
        # Load the audio file
        audio, file_sr = sf.read(file_path, always_2d=True, dtype='float32')
        debug_print(f"File loaded: {len(audio)} samples, {file_sr}Hz, {audio.shape[1]} channels", 1)
        
        # Get device info
        try:
            device_info = sd.query_devices(output_12_id)
            device_sr = int(device_info['default_samplerate'])
            debug_print(f"Device sample rate: {device_sr}Hz", 1)
        except:
            device_sr = 44100  # Default if query fails
            debug_print(f"Using default sample rate: {device_sr}Hz", 1)
        
        # Resample if needed
        if file_sr != device_sr:
            debug_print(f"Sample rate mismatch: file={file_sr}Hz, device={device_sr}Hz", 1)
            audio = simple_resample(audio, file_sr, device_sr)
        
        # Split channels
        left_channel = audio[:, 0].copy()
        right_channel = audio[:, 1].copy()
        
        # Create test tones for beginning of audio to verify sync
        test_duration = 5  # seconds
        test_samples = int(test_duration * device_sr)
        
        t = np.linspace(0, test_duration, test_samples, endpoint=False)
        test_left = 0.7 * np.sin(2 * np.pi * 440 * t).astype(np.float32)  # 440 Hz (A4)
        test_right = 0.7 * np.sin(2 * np.pi * 880 * t).astype(np.float32)  # 880 Hz (A5)
        
        # Add timing pulses (brief silence) every second
        for i in range(5):
            pulse_pos = int(i * device_sr)
            if pulse_pos + 100 < len(test_left):
                test_left[pulse_pos:pulse_pos+100] = 0
                test_right[pulse_pos:pulse_pos+100] = 0
        
        # Add fade in/out
        fade_samples = int(0.1 * device_sr)
        fade_in = np.linspace(0, 1, fade_samples)
        fade_out = np.linspace(1, 0, fade_samples)
        
        test_left[:fade_samples] *= fade_in
        test_left[-fade_samples:] *= fade_out
        test_right[:fade_samples] *= fade_in
        test_right[-fade_samples:] *= fade_out
        
        # Replace beginning of channels with test tones
        if len(left_channel) > test_samples:
            left_channel[:test_samples] = test_left
            right_channel[:test_samples] = test_right
        
        # Create dual mono output for each device
        # Left channel to both left and right speakers of Output 1/2
        left_output = np.column_stack((left_channel, left_channel)).astype(np.float32)
        
        # Right channel to both left and right speakers of Output 3/4
        right_output = np.column_stack((right_channel, right_channel)).astype(np.float32)
        
        debug_print("CHANNEL ROUTING:", 1)
        debug_print("Left channel → Output 1/2 (both speakers)", 1)
        debug_print("Right channel → Output 3/4 (both speakers)", 1)
        
        # Use a barrier for precise synchronization
        sync_barrier = threading.Barrier(3)  # 2 playback threads + main thread
        
        # Threads for synchronized playback
        def play_left():
            try:
                # Wait at the barrier for sync
                debug_print("Left channel thread ready...", 1)
                sync_barrier.wait()
                
                # Record exact start time
                start_time = time.time()
                debug_print(f"Starting left channel at {start_time:.6f}", 1)
                
                # Simple non-blocking play
                sd.play(left_output, device_sr, device=output_12_id, blocking=True)
                
                debug_print("Left channel playback complete", 1)
            except Exception as e:
                debug_print(f"Error in left channel: {e}", 1)
        
        def play_right():
            try:
                # Wait at the barrier for sync
                debug_print("Right channel thread ready...", 1)
                sync_barrier.wait()
                
                # Record exact start time
                start_time = time.time()
                debug_print(f"Starting right channel at {start_time:.6f}", 1)
                
                # Simple non-blocking play
                sd.play(right_output, device_sr, device=output_34_id, blocking=True)
                
                debug_print("Right channel playback complete", 1)
            except Exception as e:
                debug_print(f"Error in right channel: {e}", 1)
        
        # Start playback
        debug_print("\nINITIATING SYNCHRONIZED PLAYBACK:")
        
        # Create and start threads
        thread_left = threading.Thread(target=play_left)
        thread_right = threading.Thread(target=play_right)
        
        thread_left.start()
        thread_right.start()
        
        # Brief countdown
        for i in range(3, 0, -1):
            debug_print(f"Starting in {i}...", 1)
            time.sleep(1)
        
        # Trigger synchronization
        debug_print("⏯️ STARTING PLAYBACK NOW!", 1)
        sync_barrier.wait()
        
        # Wait for playback to complete
        total_duration = len(audio) / device_sr
        debug_print(f"Playing {total_duration:.1f} seconds of audio...", 1)
        
        # Show progress
        start_time = time.time()
        try:
            while thread_left.is_alive() or thread_right.is_alive():
                elapsed = time.time() - start_time
                percent = min(100, int(elapsed / total_duration * 100))
                
                # Update every 5 seconds for long files
                time.sleep(5)
                debug_print(f"Playback progress: {percent}% ({elapsed:.1f}s / {total_duration:.1f}s)", 1)
        except KeyboardInterrupt:
            debug_print("Playback interrupted by user", 1)
            sd.stop()
        
        # Wait for threads to complete
        thread_left.join()
        thread_right.join()
        
        debug_print("✓ PLAYBACK COMPLETE", 1)
        return True
        
    except Exception as e:
        debug_print(f"Error: {e}", 1)
        return False

if __name__ == "__main__":
    synchronized_playback()

[03:20:04] SIMPLE SYNCHRONIZED PLAYBACK
[03:20:04]   Using devices:
[03:20:04]   Output 1/2: Device ID 21
[03:20:04]   Output 3/4: Device ID 20
[03:20:04]   Loading file: C:\Users\cogpsy-vrlab\Desktop\audio sample\participant_00_design_2ch.wav
[03:20:05]   File loaded: 80412672 samples, 48000Hz, 2 channels
[03:20:05]   Device sample rate: 44100Hz
[03:20:05]   Sample rate mismatch: file=48000Hz, device=44100Hz
[03:20:05]   Resampling audio from 48000Hz to 44100Hz...
[03:20:07]   Resampling complete. Original samples: 80412672, New samples: 73879142
[03:20:07]   CHANNEL ROUTING:
[03:20:07]   Left channel → Output 1/2 (both speakers)
[03:20:07]   Right channel → Output 3/4 (both speakers)
[03:20:07] 
INITIATING SYNCHRONIZED PLAYBACK:
[03:20:07]   Left channel thread ready...
[03:20:07]   Right channel thread ready...
[03:20:07]   Starting in 3...
[03:20:08]   Starting in 2...
[03:20:09]   Starting in 1...
[03:20:10]   ⏯️ STARTING PLAYBACK NOW!
[03:20:10]   Playing 1675.3 seconds of audio.

In [2]:
import sounddevice as sd
import soundfile as sf
import numpy as np
import time

def debug_print(message, indent=0):
    """Print debug messages with timestamps and optional indentation"""
    timestamp = time.strftime("%H:%M:%S", time.localtime())
    indent_str = "  " * indent
    print(f"[{timestamp}] {indent_str}{message}")

def find_multichannel_device():
    """Find a device that supports 4+ channels"""
    debug_print("SEARCHING FOR MULTI-CHANNEL AUDIO DEVICES:")
    
    best_device_id = None
    best_api = None
    max_channels = 0
    
    devices = sd.query_devices()
    
    # First print all output devices
    debug_print("All output devices:", 1)
    for i, device in enumerate(devices):
        if device['max_output_channels'] > 0:
            # Get API type
            api_type = "Unknown"
            if 'hostapi' in device:
                hostapi = device['hostapi']
                if hostapi == 0:
                    api_type = "MME"
                elif hostapi == 1:
                    api_type = "DirectSound"
                elif hostapi == 2:
                    api_type = "WASAPI"
                elif hostapi == 3:
                    api_type = "ASIO"
            
            debug_print(f"ID {i}: {device['name']} - {device['max_output_channels']} channels ({api_type})", 1)
            
            # Keep track of the device with most channels
            if device['max_output_channels'] >= 4:
                debug_print(f"  ✓ Supports 4+ channels", 1)
                
                # Prioritize ASIO devices, then by channel count
                is_better = False
                
                if best_api is None:
                    is_better = True
                elif best_api != "ASIO" and api_type == "ASIO":
                    is_better = True
                elif best_api == api_type and device['max_output_channels'] > max_channels:
                    is_better = True
                
                if is_better:
                    best_device_id = i
                    best_api = api_type
                    max_channels = device['max_output_channels']
    
    # Check if we found a suitable device
    if best_device_id is not None:
        debug_print(f"\nSelected multi-channel device: ID {best_device_id}", 1)
        debug_print(f"Name: {devices[best_device_id]['name']}", 1)
        debug_print(f"API: {best_api}", 1)
        debug_print(f"Channels: {max_channels}", 1)
        return best_device_id
    else:
        debug_print("\nNo suitable multi-channel device found", 1)
        return None

def create_4channel_demo(file_path, duration=10, sample_rate=44100):
    """Create a 4-channel demo file with different content in each channel"""
    debug_print(f"CREATING 4-CHANNEL DEMO (duration: {duration}s)")
    
    # Create sample times
    samples = int(duration * sample_rate)
    t = np.linspace(0, duration, samples, endpoint=False)
    
    # Create 4 channels with different content
    # Channel 1: 440 Hz sine (A4)
    channel1 = 0.5 * np.sin(2 * np.pi * 440 * t)
    
    # Channel 2: 220 Hz sine (A3)
    channel2 = 0.5 * np.sin(2 * np.pi * 220 * t)
    
    # Channel 3: 880 Hz sine (A5)
    channel3 = 0.5 * np.sin(2 * np.pi * 880 * t)
    
    # Channel 4: 1760 Hz sine (A6)
    channel4 = 0.5 * np.sin(2 * np.pi * 1760 * t)
    
    # Add announcements at the beginning of each channel
    # Create time markers so we can identify each channel
    for i in range(1, 5):
        # Create short silences at different positions for each channel
        silence_pos = int(i * sample_rate / 2)  # Every half second
        if i == 1:
            # Channel 1 has 1 short silence
            channel1[silence_pos:silence_pos+2000] = 0
        elif i == 2:
            # Channel 2 has 2 short silences
            channel2[silence_pos:silence_pos+2000] = 0
            channel2[silence_pos+4000:silence_pos+6000] = 0
        elif i == 3:
            # Channel 3 has 3 short silences
            channel3[silence_pos:silence_pos+2000] = 0
            channel3[silence_pos+4000:silence_pos+6000] = 0
            channel3[silence_pos+8000:silence_pos+10000] = 0
        elif i == 4:
            # Channel 4 has 4 short silences
            channel4[silence_pos:silence_pos+2000] = 0
            channel4[silence_pos+4000:silence_pos+6000] = 0
            channel4[silence_pos+8000:silence_pos+10000] = 0
            channel4[silence_pos+12000:silence_pos+14000] = 0
    
    # Combine channels
    audio = np.column_stack((channel1, channel2, channel3, channel4)).astype(np.float32)
    
    # Add fade in/out
    fade_samples = int(0.1 * sample_rate)
    fade_in = np.linspace(0, 1, fade_samples)
    fade_out = np.linspace(1, 0, fade_samples)
    
    for i in range(4):
        audio[:fade_samples, i] *= fade_in
        audio[-fade_samples:, i] *= fade_out
    
    # Save the file
    output_path = file_path
    sf.write(output_path, audio, sample_rate)
    
    debug_print(f"Created 4-channel demo file: {output_path}", 1)
    debug_print("Channel 1: 440 Hz (A4) with 1 silence marker", 1)
    debug_print("Channel 2: 220 Hz (A3) with 2 silence markers", 1)
    debug_print("Channel 3: 880 Hz (A5) with 3 silence markers", 1)
    debug_print("Channel 4: 1760 Hz (A6) with 4 silence markers", 1)
    
    return output_path

def convert_stereo_to_quad(stereo_path, quad_path=None):
    """Convert a stereo file to a 4-channel file for multi-channel output"""
    debug_print(f"CONVERTING STEREO TO 4-CHANNEL: {stereo_path}")
    
    # Generate output path if not provided
    if quad_path is None:
        quad_path = stereo_path.replace(".wav", "_4ch.wav")
        if quad_path == stereo_path:
            quad_path = stereo_path + "_4ch.wav"
    
    try:
        # Load the stereo file
        audio, sr = sf.read(stereo_path, always_2d=True)
        debug_print(f"Loaded stereo file: {audio.shape[1]} channels, {sr} Hz", 1)
        
        if audio.shape[1] != 2:
            debug_print(f"Error: Input file has {audio.shape[1]} channels, expected 2", 1)
            return None
        
        # Extract the stereo channels
        left_channel = audio[:, 0]
        right_channel = audio[:, 1]
        
        # Create a 4-channel array
        # Channel mapping:
        # 1 (index 0): Left channel to Output 1/2 left speaker
        # 2 (index 1): Left channel to Output 1/2 right speaker
        # 3 (index 2): Right channel to Output 3/4 left speaker
        # 4 (index 3): Right channel to Output 3/4 right speaker
        quad_audio = np.column_stack((
            left_channel,   # Ch1: Output 1/2 left 
            left_channel,   # Ch2: Output 1/2 right
            right_channel,  # Ch3: Output 3/4 left
            right_channel   # Ch4: Output 3/4 right
        )).astype(np.float32)
        
        # Add test tones at the beginning to verify channel routing
        test_duration = 5  # seconds
        test_samples = int(test_duration * sr)
        
        if len(quad_audio) > test_samples:
            # Create time base for test tones
            t = np.linspace(0, test_duration, test_samples, endpoint=False)
            
            # Different tones for each channel
            test_ch1 = 0.5 * np.sin(2 * np.pi * 440 * t)  # 440 Hz (A4)
            test_ch2 = 0.5 * np.sin(2 * np.pi * 220 * t)  # 220 Hz (A3)
            test_ch3 = 0.5 * np.sin(2 * np.pi * 880 * t)  # 880 Hz (A5)
            test_ch4 = 0.5 * np.sin(2 * np.pi * 1760 * t)  # 1760 Hz (A6)
            
            # Add timing markers (silences) to identify each channel
            for i in range(1, 5):
                marker_pos = int(i * sr)  # Every second
                if marker_pos + 1000 < test_samples:
                    test_ch1[marker_pos:marker_pos+1000] = 0  # 1 marker
                    
                    if i >= 2:  # 2+ markers
                        test_ch2[marker_pos:marker_pos+1000] = 0
                    
                    if i >= 3:  # 3+ markers
                        test_ch3[marker_pos:marker_pos+1000] = 0
                    
                    if i >= 4:  # 4 markers
                        test_ch4[marker_pos:marker_pos+1000] = 0
            
            # Add fade in/out
            fade_samples = int(0.1 * sr)
            fade_in = np.linspace(0, 1, fade_samples)
            fade_out = np.linspace(1, 0, fade_samples)
            
            test_ch1[:fade_samples] *= fade_in
            test_ch1[-fade_samples:] *= fade_out
            test_ch2[:fade_samples] *= fade_in
            test_ch2[-fade_samples:] *= fade_out
            test_ch3[:fade_samples] *= fade_in
            test_ch3[-fade_samples:] *= fade_out
            test_ch4[:fade_samples] *= fade_in
            test_ch4[-fade_samples:] *= fade_out
            
            # Replace beginning with test tones
            quad_audio[:test_samples, 0] = test_ch1
            quad_audio[:test_samples, 1] = test_ch2  
            quad_audio[:test_samples, 2] = test_ch3
            quad_audio[:test_samples, 3] = test_ch4
        
        # Save the 4-channel file
        sf.write(quad_path, quad_audio, sr)
        
        debug_print(f"Created 4-channel file: {quad_path}", 1)
        debug_print(f"Channel mapping:", 1)
        debug_print(f"  Channel 1: Left channel → Output 1/2 left speaker", 1)
        debug_print(f"  Channel 2: Left channel → Output 1/2 right speaker", 1)
        debug_print(f"  Channel 3: Right channel → Output 3/4 left speaker", 1)
        debug_print(f"  Channel 4: Right channel → Output 3/4 right speaker", 1)
        
        return quad_path
        
    except Exception as e:
        debug_print(f"Error converting file: {e}", 1)
        return None

def play_multichannel_file(file_path, device_id=None):
    """Play a multi-channel audio file through a multi-channel device"""
    debug_print(f"PLAYING MULTI-CHANNEL FILE: {file_path}")
    
    try:
        # Load the audio file
        audio, sr = sf.read(file_path, always_2d=True)
        debug_print(f"Loaded file: {audio.shape[1]} channels, {sr} Hz, {len(audio)/sr:.1f} seconds", 1)
        
        # Check the number of channels
        num_channels = audio.shape[1]
        debug_print(f"File has {num_channels} channels", 1)
        
        # Find a suitable device if not specified
        if device_id is None:
            device_id = find_multichannel_device()
            if device_id is None:
                debug_print("No suitable multi-channel device found", 1)
                return False
        
        # Get device info
        device_info = sd.query_devices(device_id)
        debug_print(f"Using device: {device_info['name']}", 1)
        
        # Check if device supports enough channels
        device_channels = device_info['max_output_channels']
        debug_print(f"Device supports {device_channels} output channels", 1)
        
        if device_channels < num_channels:
            debug_print(f"Warning: Device only supports {device_channels} channels, but file has {num_channels}", 1)
            debug_print("Some channels will be dropped", 1)
        
        # Check sample rate
        device_sr = int(device_info['default_samplerate'])
        debug_print(f"Device sample rate: {device_sr} Hz", 1)
        
        # Start playback
        debug_print("\nSTARTING PLAYBACK:")
        for i in range(3, 0, -1):
            debug_print(f"Starting in {i}...", 1)
            time.sleep(1)
        
        debug_print("⏯️ PLAYING NOW!", 1)
        
        # Play the audio
        sd.play(audio, sr, device=device_id, blocking=False)
        
        # Monitor playback
        total_duration = len(audio) / sr
        start_time = time.time()
        
        try:
            while sd.get_stream().active:
                elapsed = time.time() - start_time
                percent = min(100, int(elapsed / total_duration * 100))
                
                # Update every 5 seconds for long files
                time.sleep(5)
                debug_print(f"Playback progress: {percent}% ({elapsed:.1f}s / {total_duration:.1f}s)", 1)
        except KeyboardInterrupt:
            debug_print("Playback interrupted by user", 1)
            sd.stop()
        
        debug_print("Playback complete!", 1)
        return True
        
    except Exception as e:
        debug_print(f"Error during playback: {e}", 1)
        return False

def multichannel_player():
    """Main function for multi-channel audio playback"""
    debug_print("MULTI-CHANNEL AUDIO PLAYER")
    debug_print("=========================")
    
    # Ask what the user wants to do
    debug_print("\nOPTIONS:")
    debug_print("1. Create a 4-channel demo file", 1)
    debug_print("2. Convert existing stereo file to 4-channel", 1)
    debug_print("3. Play an existing multi-channel file", 1)
    
    try:
        choice = int(input("\nEnter your choice (1-3): "))
        
        if choice == 1:
            # Create a demo file
            demo_path = input("Enter path for demo file [demo_4ch.wav]: ") or "demo_4ch.wav"
            duration = int(input("Enter duration in seconds [10]: ") or "10")
            
            demo_file = create_4channel_demo(demo_path, duration)
            
            # Play the demo
            if input("\nPlay the demo now? (y/n) [y]: ").lower() != 'n':
                play_multichannel_file(demo_file)
            
        elif choice == 2:
            # Convert a stereo file
            stereo_path = input("Enter path to stereo file: ")
            if not stereo_path:
                debug_print("No file specified", 1)
                return
            
            quad_path = input("Enter path for output 4-channel file [auto]: ") or None
            
            quad_file = convert_stereo_to_quad(stereo_path, quad_path)
            
            if quad_file:
                # Play the converted file
                if input("\nPlay the converted file now? (y/n) [y]: ").lower() != 'n':
                    play_multichannel_file(quad_file)
            
        elif choice == 3:
            # Play a multi-channel file
            file_path = input("Enter path to multi-channel file: ")
            if not file_path:
                debug_print("No file specified", 1)
                return
            
            play_multichannel_file(file_path)
            
        else:
            debug_print("Invalid choice", 1)
            
    except ValueError:
        debug_print("Invalid input", 1)
    except Exception as e:
        debug_print(f"Error: {e}", 1)

if __name__ == "__main__":
    multichannel_player()

[03:26:11] MULTI-CHANNEL AUDIO PLAYER
[03:26:11] =========================
[03:26:11] 
OPTIONS:
[03:26:11]   1. Create a 4-channel demo file
[03:26:11]   2. Convert existing stereo file to 4-channel
[03:26:11]   3. Play an existing multi-channel file



Enter your choice (1-3):  3
Enter path to multi-channel file:  C:\Users\cogpsy-vrlab\Desktop\audio sample\participant_00_design_4ch.wav


[03:26:28] PLAYING MULTI-CHANNEL FILE: C:\Users\cogpsy-vrlab\Desktop\audio sample\participant_00_design_4ch.wav
[03:26:29]   Loaded file: 4 channels, 44100 Hz, 1675.3 seconds
[03:26:29]   File has 4 channels
[03:26:29] SEARCHING FOR MULTI-CHANNEL AUDIO DEVICES:
[03:26:29]   All output devices:
[03:26:29]   ID 5: Microsoft Sound Mapper - Output - 2 channels (MME)
[03:26:29]   ID 6: Output 3/4 (2- Komplete Audio 6 - 2 channels (MME)
[03:26:29]   ID 7: Output 1/2 (2- Komplete Audio 6 - 2 channels (MME)
[03:26:29]   ID 8: Speakers (Realtek(R) Audio) - 2 channels (MME)
[03:26:29]   ID 9: SPDIF Output (2- Komplete Audio - 2 channels (MME)
[03:26:29]   ID 15: Primary Sound Driver - 2 channels (DirectSound)
[03:26:29]   ID 16: Output 3/4 (2- Komplete Audio 6 MK2) - 2 channels (DirectSound)
[03:26:29]   ID 17: Output 1/2 (2- Komplete Audio 6 MK2) - 2 channels (DirectSound)
[03:26:29]   ID 18: Speakers (Realtek(R) Audio) - 2 channels (DirectSound)
[03:26:29]   ID 19: SPDIF Output (2- Komplete Au

In [10]:
# ╔════════════════════════════════════════════════════════════════════════════╗
# ║  Play a 4-channel WAV on Komplete Audio 6 using TWO 2-channel ASIO outs   ║
# ╚════════════════════════════════════════════════════════════════════════════╝
import time, numpy as np, sounddevice as sd, soundfile as sf

# ─────── 1. EDIT THESE THREE LINES ────────────────────────────────────────────
WAV_PATH = r"C:\Users\cogpsy-vrlab\Desktop\audio sample\participant_00_design_4ch.wav"
DEV_12   = 42          # ← ID for “Output 1/2 (ASIO)”
DEV_34   = 41          # ← ID for “Output 3/4 (ASIO)”
BLOCK    = 1024        # frames per chunk  (leave as-is unless glitchy)
# ──────────────────────────────────────────────────────────────────────────────

# helper for timestamped prints
def dbg(msg, i=0):
    print(f"[{time.strftime('%H:%M:%S')}] {'  '*i}{msg}")

# 2) List all output devices so you can confirm the IDs
dbg("AVAILABLE OUTPUT DEVICES:")
for i, d in enumerate(sd.query_devices()):
    if d['max_output_channels'] > 0:
        api = ('MME','DirectSound','WASAPI','ASIO')[d['hostapi']]
        dbg(f"ID {i:<3}{d['name'][:50]:<50}{d['max_output_channels']} ch ({api})", 1)

# 3) Load the 4-channel file
dbg(f"LOADING {WAV_PATH!r}")
audio, sr = sf.read(WAV_PATH, always_2d=True)
if audio.shape[1] != 4:
    raise ValueError("File must have exactly 4 channels")
pair12 = audio[:, :2].astype('float32')   # channels 1-2
pair34 = audio[:, 2:].astype('float32')   # channels 3-4
total  = len(audio)
dbg(f"File: 4 ch • {sr} Hz • {total/sr:.1f} s", 1)

# 4) Open two independent 2-channel streams
dbg("▶ STARTING PLAYBACK", 1)
with sd.OutputStream(device=DEV_12, samplerate=sr, channels=2, blocksize=BLOCK,
                     dtype='float32') as s12, \
     sd.OutputStream(device=DEV_34, samplerate=sr, channels=2, blocksize=BLOCK,
                     dtype='float32') as s34:

    pos = 0
    last_pct = -1
    while pos < total:
        end = min(pos + BLOCK, total)
        s12.write(pair12[pos:end])
        s34.write(pair34[pos:end])
        pos = end

        # progress message every ~5 %
        pct = int(pos / total * 100)
        if pct // 5 != last_pct // 5:
            dbg(f" … {pct:2d}% ({pos/sr:6.1f} s / {total/sr:.1f} s)", 1)
            last_pct = pct

dbg("✅ PLAYBACK FINISHED", 1)


[03:36:10] AVAILABLE OUTPUT DEVICES:
[03:36:10]   ID 5  Microsoft Sound Mapper - Output                   2 ch (MME)
[03:36:10]   ID 6  Output 3/4 (2- Komplete Audio 6                   2 ch (MME)
[03:36:10]   ID 7  Output 1/2 (2- Komplete Audio 6                   2 ch (MME)
[03:36:10]   ID 8  Speakers (Realtek(R) Audio)                       2 ch (MME)
[03:36:10]   ID 9  SPDIF Output (2- Komplete Audio                   2 ch (MME)
[03:36:10]   ID 15 Primary Sound Driver                              2 ch (DirectSound)
[03:36:10]   ID 16 Output 3/4 (2- Komplete Audio 6 MK2)              2 ch (DirectSound)
[03:36:10]   ID 17 Output 1/2 (2- Komplete Audio 6 MK2)              2 ch (DirectSound)
[03:36:10]   ID 18 Speakers (Realtek(R) Audio)                       2 ch (DirectSound)
[03:36:10]   ID 19 SPDIF Output (2- Komplete Audio 6 MK2)            2 ch (DirectSound)
[03:36:10]   ID 20 Output 3/4 (2- Komplete Audio 6 MK2)              2 ch (WASAPI)
[03:36:10]   ID 21 Output 1/2 (2- Komple

PortAudioError: Error opening OutputStream: Unanticipated host error [PaErrorCode -9999]: 'Blocking API not supported yet' [Windows WDM-KS error -9999]